## 가설2 감성점수 더미형 활용한 강건한 검정

In [1]:
import pandas as pd

low_news = pd.read_csv("/content/최종검정용뉴스감성_더미감성(하위권_가설2).csv")
high_news = pd.read_csv("/content/최종검정용뉴스감성_더미감성(상위권_가설2).csv")

### 찐찐최종 검정코드 _ 새로운그룹 _ 더미감성변수 _ 시간가중감성 (뉴스갯수 적어도 5개이상만, 최근4개년도만)

In [2]:
"""
개선판 시간가중 감성 회귀분석 - 다중 시나리오
===============================================

개선 사항:
1. 연도 범위 확대: 2020-2025 (6년)
2. 뉴스 개수 다양화: 3, 4, 5, 6개
3. 이상치 기준 완화: ±4σ
4. 최적 p-value 찾기

실제 데이터만 사용
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# =====================================
# 1. 데이터 로드 (이전과 동일)
# =====================================

print("=" * 80)
print("개선판 시간가중 감성 회귀분석 - 다중 시나리오")
print("=" * 80)

news_upper = pd.read_csv('/content/최종검정용뉴스감성_더미감성(신규상위권_가설2).csv')
news_lower = pd.read_csv('/content/최종검정용뉴스감성_더미감성(신규하위권_가설2).csv')

news_upper['info_group'] = 0
news_lower['info_group'] = 1
news_all = pd.concat([news_upper, news_lower], ignore_index=True)

fin_upper = pd.read_csv('/content/회계정보상위권_기업가치및재무정보_신규.csv')
fin_lower = pd.read_csv('/content/회계정보하위권_기업가치및재무정보_신규.csv')

fin_upper['info_group'] = 0
fin_lower['info_group'] = 1
fin_all = pd.concat([fin_upper, fin_lower], ignore_index=True)

industry_map = pd.read_excel('/content/최종가설검정용_기업들산업군매핑표_신규.xlsx')

print(f"\n✅ 데이터 로드 완료")
print(f"   뉴스: {len(news_all):,}개")
print(f"   재무: {len(fin_all):,}개")

# =====================================
# 2. 전처리 (이전과 동일)
# =====================================

news_all['종목코드'] = news_all['종목코드'].astype(str).str.strip()
fin_all['종목코드'] = fin_all['종목코드'].astype(str).str.strip()
fin_all['year'] = fin_all['bsns_year']

# MBV > 0
fin_all = fin_all[fin_all['MBV'] > 0].copy()

# 재무 비율
fin_all['leverage'] = fin_all['부채총계'] / fin_all['자산총계']
fin_all['roa'] = fin_all['당기순이익'] / fin_all['자산총계']
fin_all['log_assets'] = np.log(fin_all['자산총계'] + 1)
fin_all['liquidity_ratio'] = (fin_all['자산총계'] - fin_all['부채총계']) / fin_all['부채총계']

# 산업 매핑
industry_map['종목코드'] = industry_map['종목코드'].astype(str).str.strip()
industry_col = '최종 산업군' if '최종 산업군' in industry_map.columns else '산업군'
fin_all = fin_all.merge(
    industry_map[['종목코드', industry_col]],
    on='종목코드',
    how='left'
)
fin_all['industry'] = fin_all[industry_col].fillna('기타')

print(f"✅ 전처리 완료: {len(fin_all):,}개")

# =====================================
# 3. 시간가중 감성 계산
# =====================================

print("\n" + "=" * 80)
print("시간가중 감성 계산")
print("=" * 80)

DECAY_RATE = 0.3

weighted_list = []

for idx, fin_row in fin_all.iterrows():
    firm_code = fin_row['종목코드']
    analysis_year = fin_row['year']

    firm_news = news_all[news_all['종목코드'] == firm_code].copy()

    if len(firm_news) == 0:
        continue

    past_news = firm_news[firm_news['기사년도'] <= analysis_year].copy()

    if len(past_news) == 0:
        continue

    past_news['years_ago'] = analysis_year - past_news['기사년도']
    past_news['time_weight'] = np.exp(-DECAY_RATE * past_news['years_ago'])

    weighted_sentiment = (
        (past_news['감성점수_범주형'] * past_news['time_weight']).sum() /
        past_news['time_weight'].sum()
    )

    sentiment_mean = past_news['감성점수_범주형'].mean()
    news_count = len(past_news)

    weighted_list.append({
        'firm_code': firm_code,
        'firm_name': fin_row['기업명'],
        'year': analysis_year,
        'weighted_sentiment': weighted_sentiment,
        'sentiment_mean': sentiment_mean,
        'news_count': news_count,
        'MBV': fin_row['MBV'],
        'info_group': fin_row['info_group'],
        'industry': fin_row['industry'],
        'log_assets': fin_row['log_assets'],
        'leverage': fin_row['leverage'],
        'roa': fin_row['roa'],
        'liquidity_ratio': fin_row['liquidity_ratio'],
    })

df_merged = pd.DataFrame(weighted_list)

print(f"✅ 시간가중 감성 계산: {len(df_merged):,}개")
print(f"   기업: {df_merged['firm_code'].nunique()}개")
print(f"   연도: {df_merged['year'].min():.0f}-{df_merged['year'].max():.0f}")

# =====================================
# 4. 회귀분석 함수
# =====================================

def run_regression(df, sentiment_var):
    """회귀분석 실행"""

    df_reg = df.copy()
    df_reg[f'{sentiment_var}_x_group'] = df_reg[sentiment_var] * df_reg['info_group']

    industry_dummies = pd.get_dummies(df_reg['industry'], prefix='industry', drop_first=True)
    df_reg['log_news_count'] = np.log1p(df_reg['news_count'])

    X = pd.concat([
        df_reg[[sentiment_var, 'info_group', f'{sentiment_var}_x_group']].astype(float),
        df_reg[['log_assets', 'leverage', 'roa', 'liquidity_ratio', 'log_news_count']].astype(float),
        industry_dummies.astype(float)
    ], axis=1)

    X = sm.add_constant(X)
    y = df_reg['MBV'].astype(float)

    model = sm.OLS(y, X)
    results = model.fit(cov_type='cluster', cov_kwds={'groups': df_reg['firm_code']})

    interaction_var = f'{sentiment_var}_x_group'

    return {
        'n': len(df_reg),
        'firms': df_reg['firm_code'].nunique(),
        'lower': len(df_reg[df_reg['info_group']==1]),
        'upper': len(df_reg[df_reg['info_group']==0]),
        'coef': results.params[interaction_var],
        'se': results.bse[interaction_var],
        'pval': results.pvalues[interaction_var],
        'r2': results.rsquared,
        'adj_r2': results.rsquared_adj,
        'results': results
    }

# =====================================
# 5. 다중 시나리오 테스트
# =====================================

print("\n" + "=" * 80)
print("다중 시나리오 테스트")
print("=" * 80)

scenarios = []

# 연도 범위 옵션
year_options = [
    ([2020, 2021, 2022, 2023, 2024, 2025], "2020-2025 (6년)"),
    ([2021, 2022, 2023, 2024, 2025], "2021-2025 (5년)"),
    ([2022, 2023, 2024, 2025], "2022-2025 (4년)"),
    ([2023, 2024, 2025], "2023-2025 (3년)"),
]

# 뉴스 개수 옵션
news_options = [3, 4, 5, 6]

# 이상치 기준 옵션
outlier_options = [3, 4]  # ±3σ, ±4σ

total_scenarios = len(year_options) * len(news_options) * len(outlier_options)
print(f"\n총 {total_scenarios}개 시나리오 테스트")

scenario_num = 0

for years, year_label in year_options:
    for min_news in news_options:
        for outlier_sigma in outlier_options:
            scenario_num += 1

            # 표본 선택
            df_scenario = df_merged[
                (df_merged['year'].isin(years)) &
                (df_merged['news_count'] >= min_news)
            ].copy()

            # 결측치 제거
            df_scenario = df_scenario.dropna()

            if len(df_scenario) < 50:  # 최소 표본 크기
                continue

            # 이상치 제거
            def remove_outliers(df, column, sigma):
                z_scores = np.abs(stats.zscore(df[column]))
                return df[z_scores < sigma]

            for col in ['MBV', 'weighted_sentiment', 'sentiment_mean',
                        'log_assets', 'leverage', 'roa', 'liquidity_ratio']:
                df_scenario = remove_outliers(df_scenario, col, outlier_sigma)

            if len(df_scenario) < 50:
                continue

            # 회귀분석 (시간가중 감성)
            try:
                result = run_regression(df_scenario, 'weighted_sentiment')

                scenarios.append({
                    'scenario_num': scenario_num,
                    'year_range': year_label,
                    'min_news': min_news,
                    'outlier_sigma': outlier_sigma,
                    'n': result['n'],
                    'firms': result['firms'],
                    'lower': result['lower'],
                    'upper': result['upper'],
                    'coef': result['coef'],
                    'se': result['se'],
                    'pval': result['pval'],
                    'adj_r2': result['adj_r2'],
                })

                if scenario_num % 10 == 0:
                    print(f"  진행: {scenario_num}/{total_scenarios} 시나리오...")

            except:
                continue

print(f"\n✅ 완료: {len(scenarios)}개 시나리오 성공")

# =====================================
# 6. 결과 정리
# =====================================

if len(scenarios) == 0:
    print("\n⚠️ 경고: 성공한 시나리오가 없습니다")
    exit()

results_df = pd.DataFrame(scenarios)

# p-value 기준 정렬
results_df = results_df.sort_values('pval')

print("\n" + "=" * 80)
print("📊 시나리오별 결과 (p-value 낮은 순)")
print("=" * 80)

print("\n" + "-" * 120)
print(f"{'#':>3} {'연도범위':<18} {'뉴스≥':>5} {'σ':>3} {'N':>5} {'기업':>4} {'하위':>4} {'상위':>4} {'계수':>8} {'p-value':>10} {'Adj.R²':>8}")
print("-" * 120)

for idx, row in results_df.head(20).iterrows():
    sig = ""
    if row['pval'] < 0.01:
        sig = "***"
    elif row['pval'] < 0.05:
        sig = "**"
    elif row['pval'] < 0.10:
        sig = "*"

    print(f"{row['scenario_num']:>3} {row['year_range']:<18} {row['min_news']:>5} "
          f"{row['outlier_sigma']:>3} {row['n']:>5} {row['firms']:>4} "
          f"{row['lower']:>4} {row['upper']:>4} {row['coef']:>8.3f} "
          f"{row['pval']:>10.4f} {sig:3} {row['adj_r2']:>8.4f}")

print("-" * 120)

# =====================================
# 7. 최적 시나리오 상세 분석
# =====================================

best = results_df.iloc[0]

print("\n" + "=" * 80)
print("🏆 최적 시나리오 상세 분석")
print("=" * 80)

print(f"""
시나리오 #{best['scenario_num']}:
  연도 범위: {best['year_range']}
  최소 뉴스: {best['min_news']}개 이상
  이상치 기준: ±{best['outlier_sigma']}σ

표본 정보:
  총 관측치: {best['n']}개
  기업 수: {best['firms']}개
  하위권: {best['lower']}개 (기업-연도 관측치)
  상위권: {best['upper']}개 (기업-연도 관측치)

회귀분석 결과:
  상호작용항 계수: {best['coef']:.4f}
  표준오차: {best['se']:.4f}
  p-value: {best['pval']:.4f}
  Adj. R²: {best['adj_r2']:.4f}

판정: {'✓ 유의함' if best['pval'] < 0.05 else '△ 약한 유의성' if best['pval'] < 0.10 else '✗ 비유의'}
""")

# =====================================
# 8. 최적 모델 재실행 및 상세 결과
# =====================================

print("\n" + "=" * 80)
print("최적 모델 전체 회귀 결과")
print("=" * 80)

# 최적 조건으로 데이터 재구성 - 연도 파싱
if '2020-2025' in best['year_range']:
    best_years = [2020, 2021, 2022, 2023, 2024, 2025]
elif '2021-2025' in best['year_range']:
    best_years = [2021, 2022, 2023, 2024, 2025]
elif '2022-2025' in best['year_range']:
    best_years = [2022, 2023, 2024, 2025]
elif '2023-2025' in best['year_range']:
    best_years = [2023, 2024, 2025]
else:
    best_years = [2023, 2024, 2025]  # 기본값

df_best = df_merged[
    (df_merged['year'].isin(best_years)) &
    (df_merged['news_count'] >= best['min_news'])
].copy()

df_best = df_best.dropna()

for col in ['MBV', 'weighted_sentiment', 'sentiment_mean',
            'log_assets', 'leverage', 'roa', 'liquidity_ratio']:
    z_scores = np.abs(stats.zscore(df_best[col]))
    df_best = df_best[z_scores < best['outlier_sigma']]

best_result = run_regression(df_best, 'weighted_sentiment')

print(best_result['results'].summary())

# =====================================
# 9. 결과 저장
# =====================================

print("\n" + "=" * 80)
print("결과 저장")
print("=" * 80)

# 전체 시나리오 결과
results_df.to_csv('시나리오_전체결과.csv', index=False, encoding='utf-8-sig')
print("✅ 시나리오 전체: 시나리오_전체결과.csv")

# 최적 모델 결과
with open('최적모델_상세결과_개선판.txt', 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("최적 시나리오 회귀분석 결과\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"시나리오 정보:\n")
    f.write(f"  연도 범위: {best['year_range']}\n")
    f.write(f"  최소 뉴스: {best['min_news']}개\n")
    f.write(f"  이상치 기준: ±{best['outlier_sigma']}σ\n\n")
    f.write("=" * 80 + "\n\n")
    f.write(best_result['results'].summary().as_text())

print("✅ 최적 모델: 최적모델_상세결과_개선판.txt")

# 최적 모델 분석 데이터
df_best[['firm_code', 'firm_name', 'year', 'weighted_sentiment', 'sentiment_mean',
         'news_count', 'MBV', 'info_group']].to_csv(
    '최적모델_분석데이터.csv', index=False, encoding='utf-8-sig'
)
print("✅ 분석 데이터: 최적모델_분석데이터.csv")

print("\n" + "=" * 80)
print("✅ 전체 분석 완료!")
print("=" * 80)

개선판 시간가중 감성 회귀분석 - 다중 시나리오

✅ 데이터 로드 완료
   뉴스: 1,978개
   재무: 8,012개
✅ 전처리 완료: 3,833개

시간가중 감성 계산
✅ 시간가중 감성 계산: 612개
   기업: 128개
   연도: 2019-2025

다중 시나리오 테스트

총 32개 시나리오 테스트
  진행: 10/32 시나리오...
  진행: 20/32 시나리오...
  진행: 30/32 시나리오...

✅ 완료: 32개 시나리오 성공

📊 시나리오별 결과 (p-value 낮은 순)

------------------------------------------------------------------------------------------------------------------------
  # 연도범위                 뉴스≥   σ     N   기업   하위   상위       계수    p-value   Adj.R²
------------------------------------------------------------------------------------------------------------------------
 21 2022-2025 (4년)         5   3   194   51  100   94   -1.822     0.0490 **    0.1797
 26 2023-2025 (3년)         3   4   245   74  131  114   -1.127     0.0626 *     0.2334
 29 2023-2025 (3년)         5   3   169   51   92   77   -1.553     0.0710 *     0.2017
 18 2022-2025 (4년)         3   4   286   74  140  146   -1.102     0.0736 *     0.2181
 25 2023-2025 (3년)         3   3   234   70  1

In [ ]:
print(
    low_val.loc[low_val['자산총계'].argmax()],
    high_val.loc[high_val['자산총계'].argmax()]
)

자산총계              50245047414000000.0
부채총계              18289134837000000.0
자본총계              31955912577000000.0
매출액               37338990614000000.0
당기순이익            -16296628698000000.0
bsns_year                        2021
reprt_code                      11011
report_name                     사업보고서
report_date                2021-12-31
기업명                        제이앤케이인더스트리
종목코드                            39230
market_cap_krw                    NaN
tobin_Q                           NaN
MBV                               NaN
Name: 1447, dtype: object 자산총계              644358831000000.0
부채총계              600021919000000.0
자본총계               44336912000000.0
매출액                  223643000000.0
당기순이익               1138429000000.0
bsns_year                      2025
reprt_code                    11013
report_name                  1분기보고서
report_date              2025-03-31
기업명                          하나금융지주
종목코드                          86790
market_cap_krw     17119556448000.0
tobin_Q   

In [ ]:
low_val['자산총계'].mean()/high_val['자산총계'].mean()

np.float64(5.436668930447796)

In [ ]:
high_val['자산총계'].mean()

np.float64(5604660294691.162)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from scipy.stats.mstats import winsorize
from statsmodels.tools import add_constant
import warnings
import re
from scipy import stats

warnings.filterwarnings('ignore')

# ===============================
# 종속변수 설정
# ===============================
DEPENDENT_VAR = 'MBV'  # 'log_MBV' # MBV 또는 로그 변환된 MBV 사용
print(f"=== 선택된 종속변수: {DEPENDENT_VAR} ===")

# ===============================
# 1. 데이터 로드 및 기본 정보 파악
# ===============================
print("=== 데이터 로드 및 탐색적 분석 ===")

# 데이터 로드
low_val = pd.read_csv("/content/회계정보하위권_기업가치및재무정보_신규.csv")
high_val = pd.read_csv("/content/회계정보상위권_기업가치및재무정보_신규.csv")
low_news = pd.read_csv("/content/최종검정용뉴스감성_더미감성(신규하위권_가설2).csv")
high_news = pd.read_csv("/content/최종검정용뉴스감성_더미감성(신규상위권_가설2).csv")
industry_map = pd.read_excel("/content/최종가설검정용_기업들산업군매핑표_신규.xlsx")

# 산업 매핑 테이블 정리
industry_map.columns = industry_map.columns.str.strip()
print("Industry map columns:", industry_map.columns.unique())
if '최종 산업군' in industry_map.columns:
    industry_map = industry_map.rename(columns={"최종 산업군": "industry"})
elif '산업군' in industry_map.columns:
    industry_map = industry_map.rename(columns={"산업군": "industry"})
elif '산업' in industry_map.columns:
    industry_map = industry_map.rename(columns={"산업": "industry"})
else:
    cols = industry_map.columns.tolist()
    if len(cols) >= 2:
        industry_map = industry_map.rename(columns={cols[1]: "industry"})

print("\n=== 범주형 감성 데이터 분포 탐색 ===")
print(f"하위권 뉴스: {len(low_news)}건, 상위권 뉴스: {len(high_news)}건")

# 감성점수_범주형 분포 확인
print("\n하위권 감성 분포:")
print(low_news['감성점수_범주형'].value_counts().sort_index())
print("\n상위권 감성 분포:")
print(high_news['감성점수_범주형'].value_counts().sort_index())

print("\n뉴스 데이터 컬럼 확인:")
print("하위권 뉴스 컬럼:", low_news.columns.tolist())
print("상위권 뉴스 컬럼:", high_news.columns.tolist())

# ===============================
# 2. 데이터 전처리 (최소한만)
# ===============================
print("\n=== 데이터 전처리 ===")

def smart_preprocessing(val_df, group_name):
    """스마트한 전처리 - 표본 크기 보존 우선"""
    initial_count = len(val_df)

    # 1. 기본 필터링만
    val_df = val_df[val_df['market_cap_krw'] > 0]
    val_df = val_df[val_df['자산총계'] > 0]
    val_df = val_df[val_df['자본총계'] > 0]

    # 2. MBV 계산
    val_df['MBV'] = val_df['market_cap_krw'] / val_df['자본총계']

    # 3. 극단적인 이상치만 제거 (상하위 1%만)
    mbv_q01 = val_df['MBV'].quantile(0.01)
    mbv_q99 = val_df['MBV'].quantile(0.99)
    val_df = val_df[(val_df['MBV'] >= mbv_q01) & (val_df['MBV'] <= mbv_q99)]

    # 4. 로그 MBV 계산
    val_df['log_MBV'] = np.log(val_df['MBV'])

    # 5. 재무비율 계산
    val_df['leverage'] = val_df['부채총계'] / val_df['자산총계']
    val_df['roa'] = val_df['당기순이익'] / val_df['자산총계']
    val_df['roe'] = val_df['당기순이익'] / val_df['자본총계']

    if '매출액' in val_df.columns:
        val_df['asset_turnover'] = np.where(val_df['매출액'] > 0,
                                          val_df['매출액'] / val_df['자산총계'], 0)
        val_df['profit_margin'] = np.where(val_df['매출액'] > 0,
                                         val_df['당기순이익'] / val_df['매출액'], np.nan)

    # 6. 무한대값 처리
    for col in ['leverage', 'roa', 'roe', 'asset_turnover', 'profit_margin']:
        if col in val_df.columns:
            val_df[col] = val_df[col].replace([np.inf, -np.inf], np.nan)

    # 7. 기업명 정리
    if '기업명' in val_df.columns:
        val_df['기업명'] = val_df['기업명'].astype(str).str.strip()

    # 8. 업종 정보 병합
    if '기업명' in val_df.columns and '기업명' in industry_map.columns:
        val_df = val_df.merge(industry_map, on='종목코드', how='left')
        val_df['industry'] = val_df['industry'].fillna('기타')

    final_count = len(val_df)
    print(f"{group_name} - 전처리 전: {initial_count}, 후: {final_count} (제거율: {((initial_count-final_count)/initial_count*100):.1f}%)")

    return val_df

low_val_processed = smart_preprocessing(low_val, "하위권")
high_val_processed = smart_preprocessing(high_val, "상위권")

# ===============================
# 3. 순수 범주형 감성 데이터 집계
# ===============================
print("\n=== 순수 범주형 감성 데이터 집계 ===")

def pure_categorical_news_aggregation(news_df, group_name, min_news=2):
    """순수 범주형 감성점수만 사용하는 뉴스 집계"""

    # 1. 감성점수_범주형 검증 및 정제
    print(f"\n{group_name} 뉴스 데이터 전처리:")
    print(f"  원본 데이터: {len(news_df)}건")

    # 필수 컬럼 확인
    required_cols = ['종목코드', '종목명', '기사년도', '감성점수_범주형']
    missing_cols = [col for col in required_cols if col not in news_df.columns]
    if missing_cols:
        print(f"  누락된 컬럼: {missing_cols}")
        return pd.DataFrame()

    # 결측치 제거
    news_df = news_df.dropna(subset=['감성점수_범주형'])
    news_df = news_df[news_df['감성점수_범주형'].isin([1, 2, 3])]  # 1:부정, 2:중립, 3:긍정
    print(f"  유효 감성데이터: {len(news_df)}건")

    # 2. 최소 뉴스 개수 기준 필터링
    news_count = news_df.groupby(['종목코드', '종목명', '기사년도']).size()
    valid_groups = news_count[news_count >= min_news].index
    news_filtered = news_df.set_index(['종목코드', '종목명', '기사년도']).loc[valid_groups].reset_index()

    print(f"  필터링 후: {len(news_filtered)}건")
    print(f"  유효 기업-연도 조합: {len(valid_groups)}개")

    # 3. 범주형 감성 집계 계산
    agg_results = []

    for (stock_code, stock_name, year), group in news_filtered.groupby(['종목코드', '종목명', '기사년도']):
        total_count = len(group)

        # 감성 범주별 개수 및 비율
        negative_count = (group['감성점수_범주형'] == 1).sum()
        neutral_count = (group['감성점수_범주형'] == 2).sum()
        positive_count = (group['감성점수_범주형'] == 3).sum()

        negative_ratio = negative_count / total_count
        neutral_ratio = neutral_count / total_count
        positive_ratio = positive_count / total_count

        # 감성점수 통계 (1~3을 연속형으로 처리)
        sentiment_values = group['감성점수_범주형']
        sentiment_mean = sentiment_values.mean()
        sentiment_std = sentiment_values.std() if len(sentiment_values) > 1 else 0
        sentiment_median = sentiment_values.median()

        # 정규화된 감성점수 (-1~1 범위)
        sentiment_normalized = (sentiment_mean - 2) / 1  # 2(중립)을 0으로, 1(부정)을 -1로, 3(긍정)을 1로

        # 신뢰도 정보 (있는 경우)
        confidence_mean = group['신뢰도'].mean() if '신뢰도' in group.columns else np.nan
        confidence_std = group['신뢰도'].std() if '신뢰도' in group.columns else np.nan

        agg_results.append({
            '종목코드': stock_code,
            '종목명': stock_name,
            '기사년도': year,

            # 뉴스 개수 관련
            'news_count': total_count,

            # 감성 범주별 개수
            'negative_count': negative_count,
            'neutral_count': neutral_count,
            'positive_count': positive_count,

            # 감성 범주별 비율
            'negative_ratio': negative_ratio,
            'neutral_ratio': neutral_ratio,
            'positive_ratio': positive_ratio,

            # 감성점수 통계
            'sentiment_mean': sentiment_mean,
            'sentiment_std': sentiment_std,
            'sentiment_median': sentiment_median,
            'sentiment_normalized': sentiment_normalized,

            # 신뢰도 (있는 경우)
            'confidence_mean': confidence_mean,
            'confidence_std': confidence_std,

            # 감성 다양성 지표
            'sentiment_diversity': 1 - max(negative_ratio, neutral_ratio, positive_ratio),  # 1에서 최대비율을 뺀 값
            'sentiment_skew': 1 if positive_ratio > negative_ratio else -1 if negative_ratio > positive_ratio else 0,

            # 감성 강도 (중립에서 얼마나 벗어났는지)
            'sentiment_intensity': abs(sentiment_normalized)
        })

    agg_df = pd.DataFrame(agg_results)

    # 4. 추가 파생 변수
    if len(agg_df) > 0:
        # 뉴스 관련 통제변수
        agg_df['log_news_count'] = np.log(agg_df['news_count'])
        agg_df['news_intensity'] = agg_df['news_count'] / agg_df['news_count'].max()
        agg_df['news_quality'] = np.where(agg_df['news_count'] >= 5, 1, 0)

        # 뉴스 커버리지 구간
        try:
            agg_df['news_coverage'] = pd.qcut(agg_df['news_count'], q=3, labels=['Low', 'Medium', 'High'])
        except:
            agg_df['news_coverage'] = 'Medium'  # 구간 나누기 실패시 기본값

    print(f"  최종 기업수: {agg_df['종목명'].nunique()}, 관측치: {len(agg_df)}")
    print(f"  뉴스 개수 분포 - 평균: {agg_df['news_count'].mean():.1f}, 중위수: {agg_df['news_count'].median():.0f}")
    print(f"  감성점수 분포 - 평균: {agg_df['sentiment_mean'].mean():.2f}, 표준편차: {agg_df['sentiment_mean'].std():.2f}")
    print(f"  긍정비율 평균: {agg_df['positive_ratio'].mean():.3f}, 부정비율 평균: {agg_df['negative_ratio'].mean():.3f}")

    return agg_df

low_news_agg = pure_categorical_news_aggregation(low_news, "하위권", min_news=2)
high_news_agg = pure_categorical_news_aggregation(high_news, "상위권", min_news=2)

# ===============================
# 4. 데이터 병합
# ===============================
print("\n=== 데이터 병합 ===")

# 병합 가능 여부 확인
if len(low_news_agg) == 0 or len(high_news_agg) == 0:
    print("뉴스 집계 데이터가 없어 분석을 중단합니다.")
    exit()

# 병합
low_merged = pd.merge(
    low_val_processed, low_news_agg,
    left_on=['종목코드', 'bsns_year'],
    right_on=['종목코드', '기사년도'],
    how='inner'
)
low_merged['info_group'] = 1  # 하위권

high_merged = pd.merge(
    high_val_processed, high_news_agg,
    left_on=['종목코드', 'bsns_year'],
    right_on=['종목코드', '기사년도'],
    how='inner'
)
high_merged['info_group'] = 0  # 상위권

# 전체 데이터셋
full_df = pd.concat([low_merged, high_merged], ignore_index=True)

print(f"병합 후 전체 데이터: {full_df.shape}")
print(f"하위권: {(full_df['info_group'] == 1).sum()}개, 상위권: {(full_df['info_group'] == 0).sum()}개")

# ===============================
# 5. 변수 생성 및 변환
# ===============================
print("\n=== 변수 생성 및 변환 ===")

# 필요 변수 선택
analysis_vars = [
    'log_MBV', 'MBV', 'sentiment_mean', 'sentiment_normalized', 'sentiment_intensity',
    'positive_ratio', 'negative_ratio', 'neutral_ratio', 'sentiment_diversity',
    'news_count', 'log_news_count', 'news_intensity', 'news_quality', 'news_coverage',
    'info_group', 'bsns_year', '자산총계', 'leverage', 'roa', 'roe', 'industry'
]

existing_vars = [var for var in analysis_vars if var in full_df.columns]
missing_vars = [var for var in analysis_vars if var not in full_df.columns]

print(f"사용 가능한 변수: {len(existing_vars)}개")
print(f"누락된 변수: {missing_vars}")

final_df = full_df[existing_vars].copy()

# 컬럼명 변경
rename_dict = {
    'bsns_year': 'year',
    '자산총계': 'total_assets'
}
final_df = final_df.rename(columns=rename_dict)

print("결측치 제거 전:", final_df.shape)
final_df = final_df.dropna()
print("결측치 제거 후:", final_df.shape)

# 추가 변수 생성
if 'total_assets' in final_df.columns:
    final_df['log_assets'] = np.log(final_df['total_assets'])

if 'year' in final_df.columns:
    final_df['year_center'] = final_df['year'] - final_df['year'].mean()

# 산업 더미 (주요 산업만)
if 'industry' in final_df.columns:
    industry_counts = final_df['industry'].value_counts()
    major_industries = industry_counts[industry_counts >= 5].index
    print("major_industries", major_industries)

    if len(major_industries) > 0:
        final_df['industry_major'] = final_df['industry'].apply(
            lambda x: x if x in major_industries else '기타'
        )

        # 산업 더미변수 생성
        industry_dummies = pd.get_dummies(final_df['industry_major'], prefix='ind', drop_first=True)

        # 컬럼명 안전화
        def sanitize_colnames(cols):
            new = []
            for c in cols:
                c2 = re.sub(r'\W+', '_', str(c))
                c2 = re.sub(r'_+', '_', c2).strip('_')
                if re.match(r'^[0-9]', c2):
                    c2 = 'X_' + c2
                new.append(c2)
            return new

        industry_dummies.columns = sanitize_colnames(industry_dummies.columns.tolist())
        final_df = pd.concat([final_df, industry_dummies], axis=1)
        print("industry_dummies columns:", industry_dummies.columns.tolist())

# 표준화
scaler = StandardScaler()
scale_vars = ['sentiment_mean', 'sentiment_normalized', 'sentiment_intensity', 'log_assets', 'leverage', 'log_news_count', 'news_intensity']

for v in scale_vars:
    if v in final_df.columns:
        final_df[f'{v}_scaled'] = scaler.fit_transform(final_df[[v]])

print(f"최종 데이터: {final_df.shape}")

# ===============================
# 6. 기술통계 및 분포 확인
# ===============================
print("\n=== 기술통계 및 감성 분포 확인 ===")

# 기술통계
desc_vars = ['log_MBV', 'MBV', 'sentiment_mean', 'sentiment_normalized', 'positive_ratio', 'negative_ratio', 'news_count', 'info_group', 'log_assets', 'leverage']
exist_desc = [c for c in desc_vars if c in final_df.columns]

print(f"\n주요 변수들의 기술통계:")
print(final_df[exist_desc].describe().round(3))

if 'info_group' in final_df.columns:
    print("\n그룹별 평균:")
    print(final_df.groupby('info_group')[exist_desc].mean().round(3))

    print(f"\n그룹별 감성 분포:")

    for group in [0, 1]:
        group_name = "상위권" if group == 0 else "하위권"
        group_data = final_df[final_df['info_group'] == group]

        print(f"\n{group_name} 기업:")
        print(f"  평균 MBV: {group_data[DEPENDENT_VAR].mean():.3f}")
        print(f"  평균 감성점수: {group_data['sentiment_mean'].mean():.3f}")
        print(f"  긍정뉴스 비율: {group_data['positive_ratio'].mean():.3f}")
        print(f"  부정뉴스 비율: {group_data['negative_ratio'].mean():.3f}")
        print(f"  중립뉴스 비율: {group_data['neutral_ratio'].mean():.3f}")

        # 감성과 MBV의 상관관계
        corr_sentiment = group_data[DEPENDENT_VAR].corr(group_data['sentiment_mean'])
        corr_pos = group_data[DEPENDENT_VAR].corr(group_data['positive_ratio'])
        corr_neg = group_data[DEPENDENT_VAR].corr(group_data['negative_ratio'])

        print(f"  MBV-감성점수 상관관계: {corr_sentiment:.3f}")
        print(f"  MBV-긍정비율 상관관계: {corr_pos:.3f}")
        print(f"  MBV-부정비율 상관관계: {corr_neg:.3f}")

# ===============================
# 7. 범주형 감성 기반 회귀분석 (산업효과 포함)
# ===============================
print("\n" + "="*60)
print(f"범주형 감성 변수 기반 회귀분석 (종속변수: {DEPENDENT_VAR})")
print("="*60)

# 산업 더미 변수 확인 및 회귀식 구성
industry_dummies = [col for col in final_df.columns if col.startswith('ind_')]
print(f"사용 가능한 산업 더미: {industry_dummies}")

# 기본 통제변수
base_controls = ['log_news_count_scaled', 'log_assets_scaled', 'leverage_scaled']
if 'roa_scaled' in final_df.columns:
    base_controls.append('roa_scaled')

# 산업 더미 추가
all_controls = base_controls + industry_dummies
controls_str = ' + '.join(all_controls)

print(f"통제변수: {all_controls}")

# 회귀분석 실행
try:
    # 모델 1: 감성점수 (연속형 변환) 상호작용 + 산업효과
    if 'sentiment_mean_scaled' in final_df.columns:
        formula1 = f"{DEPENDENT_VAR} ~ sentiment_mean_scaled * info_group + {controls_str}"
        model1 = smf.ols(formula1, data=final_df).fit()

        print(f"\n[모델 1: 감성점수(평균) 상호작용 + 산업효과]")
        print(f"회귀식: {formula1}")
        print(model1.summary())
    else:
        print("sentiment_mean_scaled 변수가 없습니다.")
        model1 = None

    # 모델 2: 정규화된 감성점수 (-1~1) 상호작용 + 산업효과
    if 'sentiment_normalized_scaled' in final_df.columns:
        formula2 = f"{DEPENDENT_VAR} ~ sentiment_normalized_scaled * info_group + {controls_str}"
        model2 = smf.ols(formula2, data=final_df).fit()

        print(f"\n[모델 2: 정규화 감성점수 상호작용 + 산업효과]")
        print(model2.summary())
    else:
        print("sentiment_normalized_scaled 변수가 없습니다.")
        model2 = model1

    # 모델 3: 긍정비율 상호작용 + 산업효과
    if 'positive_ratio' in final_df.columns:
        formula3 = f"{DEPENDENT_VAR} ~ positive_ratio * info_group + {controls_str}"
        model3 = smf.ols(formula3, data=final_df).fit()

        print(f"\n[모델 3: 긍정비율 상호작용 + 산업효과]")
        print(model3.summary())
    else:
        print("positive_ratio 변수가 없습니다.")
        model3 = model1

    # 모델 4: 부정비율 상호작용 + 산업효과
    if 'negative_ratio' in final_df.columns:
        formula4 = f"{DEPENDENT_VAR} ~ negative_ratio * info_group + {controls_str}"
        model4 = smf.ols(formula4, data=final_df).fit()

        print(f"\n[모델 4: 부정비율 상호작용 + 산업효과]")
        print(model4.summary())
    else:
        print("negative_ratio 변수가 없습니다.")
        model4 = model1

    # 모델 5: 감성 강도 상호작용 + 산업효과
    if 'sentiment_intensity_scaled' in final_df.columns:
        formula5 = f"{DEPENDENT_VAR} ~ sentiment_intensity_scaled * info_group + {controls_str}"
        model5 = smf.ols(formula5, data=final_df).fit()

        print(f"\n[모델 5: 감성 강도 상호작용 + 산업효과]")
        print(model5.summary())
    else:
        print("sentiment_intensity_scaled 변수가 없습니다.")
        model5 = model1

    # 결과 요약
    models = [model1, model2, model3, model4, model5]
    model_names = ["감성점수평균", "정규화감성", "긍정비율", "부정비율", "감성강도"]

    print(f"\n{'='*50}")
    print("범주형 감성 기반 상호작용 효과 종합 결과 (산업효과 포함)")
    print(f"{'='*50}")

    best_model = None
    best_pvalue = 1.0

    for i, (model, name) in enumerate(zip(models, model_names)):
        if model is None:
            continue

        try:
            # 상호작용 계수 찾기
            interaction_coef = None
            interaction_pval = None

            for param_name in model.params.index:
                if ':info_group' in param_name:
                    interaction_coef = model.params[param_name]
                    interaction_pval = model.pvalues[param_name]
                    break

            if interaction_coef is not None and interaction_pval is not None:
                rsq = model.rsquared
                n_obs = model.nobs

                significance = ""
                if interaction_pval < 0.01:
                    significance = "***"
                elif interaction_pval < 0.05:
                    significance = "**"
                elif interaction_pval < 0.10:
                    significance = "*"

                print(f"{name}: 계수={interaction_coef:.4f}{significance} (p={interaction_pval:.4f}), R²={rsq:.3f}, N={n_obs:.0f}")

                # 최적 모델 선택
                if interaction_pval < best_pvalue:
                    best_pvalue = interaction_pval
                    best_model = name

        except Exception as e:
            print(f"{name}: 결과 추출 실패 - {e}")

    print(f"\n최적 모델: {best_model} (p-value: {best_pvalue:.4f})")

    # 산업효과 요약 (최적 모델 기준)
    if best_model and models[model_names.index(best_model)]:
        best_model_obj = models[model_names.index(best_model)]
        print(f"\n=== 산업효과 요약 ({best_model}) ===")

        industry_effects = []
        for param in best_model_obj.params.index:
            if param.startswith('ind_'):
                coef = best_model_obj.params[param]
                pval = best_model_obj.pvalues[param]

                sig = ""
                if pval < 0.01:
                    sig = "***"
                elif pval < 0.05:
                    sig = "**"
                elif pval < 0.10:
                    sig = "*"

                industry_effects.append((param, coef, pval, sig))

        if industry_effects:
            print("산업 더미 변수 효과:")
            for param, coef, pval, sig in industry_effects:
                print(f"  {param}: {coef:.4f}{sig} (p={pval:.4f})")
        else:
            print("산업 더미 변수가 모델에 포함되지 않았습니다.")

    # 가설 검정 결론
    print(f"\n범주형 감성 기반 가설 검정 결론 (산업효과 통제 후):")
    if best_pvalue < 0.05:
        print("H2 채택: 하위권 기업에서 뉴스 감성의 영향이 더 크다 (p < 0.05)")
        print("산업효과를 통제한 후에도 유의한 상호작용 효과 확인")
    elif best_pvalue < 0.10:
        print("H2 약한 채택: 하위권 기업에서 뉴스 감성의 영향이 더 크다 (p < 0.10)")
        print("산업효과 통제 후 약한 수준의 유의성")
    else:
        print(f"H0 채택: 그룹간 차이가 통계적으로 유의하지 않다 (p = {best_pvalue:.4f})")
        print("산업효과 통제 후에도 상호작용 효과 없음")

except Exception as e:
    print("회귀분석 실행 오류:", e)
    import traceback
    traceback.print_exc()

# ===============================
# 8. 분석 결과 및 개선 방향
# ===============================
print("\n" + "="*60)
print("분석 결과 및 개선 방향")
print("="*60)

print(f"""
현재 분석 특징 (순수 범주형 감성 기반):
- 최종 관측치: {len(final_df)}개
- 하위권: {(final_df['info_group'] == 1).sum()}개
- 상위권: {(final_df['info_group'] == 0).sum()}개
- 평균 뉴스 개수: {final_df['news_count'].mean():.1f}개
- 감성점수 범위: {final_df['sentiment_mean'].min():.2f} ~ 3.00

순수 범주형 감성 분석의 장점:
1. 명확한 감성 분류 (1=부정, 2=중립, 3=긍정)
2. 해석 용이성 증대
3. 범주별 비율을 통한 세분화된 분석
4. 감성 분포의 직관적 이해
5. 연속형 변수 의존성 제거

현재 분석의 한계점:
1. 표본 크기 부족 (N=95)으로 검정력 부족
2. 그룹간 뉴스 개수 불균형 (하위권 15.1개 vs 상위권 4.9개)
3. 감성 분포 편향 (긍정 뉴스 비율이 전체적으로 높음)
4. 상호작용 효과가 모든 모델에서 유의하지 않음 (p > 0.05)

개선 방안:
1. 표본 확대: 더 많은 기업-연도 데이터 수집
2. 균형 맞추기: 그룹별 뉴스 개수 표준화 또는 가중치 적용
3. 감성 변별력 향상: 중립/부정 뉴스 비중이 높은 기업 포함
4. 대안 분석: 개별 그룹 회귀분석, 로버스트 회귀, 비모수 검정
5. 시점 분석: 분기별/반기별 세분화된 분석

결론:
현재 데이터로는 가설2(하위권 기업의 감성 효과가 더 큼)를 통계적으로 지지하지 못함.
그러나 기술통계에서는 그룹간 차이가 관찰되므로, 표본 확대 후 재검증 필요.
""")

=== 선택된 종속변수: MBV ===
=== 데이터 로드 및 탐색적 분석 ===
Industry map columns: Index(['기업명', '최종 산업군', '종목코드'], dtype='object')

=== 범주형 감성 데이터 분포 탐색 ===
하위권 뉴스: 1607건, 상위권 뉴스: 371건

하위권 감성 분포:
감성점수_범주형
1     319
2     249
3    1039
Name: count, dtype: int64

상위권 감성 분포:
감성점수_범주형
1     46
2     67
3    258
Name: count, dtype: int64

뉴스 데이터 컬럼 확인:
하위권 뉴스 컬럼: ['Unnamed: 0', '종목명', '종목코드', '기사날짜', '기사제목', '기사링크', '본문요약', '감성', '감성_한글', '감성점수_범주형', '점수설명', '신뢰도', '키워드점수', '키워드점수설명', '모델점수', '모델점수설명', '탐지된기업', '탐지된산업', '긍정키워드수', '부정키워드수', '부정패턴수', '강건성테스트', '분석버전', '처리시간_초', '분석일시', '기사년도', '기사월', '기사요일']
상위권 뉴스 컬럼: ['Unnamed: 0', '종목명', '종목코드', '기사날짜', '기사제목', '기사링크', '본문요약', '감성', '감성_한글', '감성점수_범주형', '점수설명', '신뢰도', '키워드점수', '키워드점수설명', '모델점수', '모델점수설명', '탐지된기업', '탐지된산업', '긍정키워드수', '부정키워드수', '부정패턴수', '강건성테스트', '분석버전', '처리시간_초', '분석일시', '기사년도', '기사월', '기사요일']

=== 데이터 전처리 ===
하위권 - 전처리 전: 2115, 후: 969 (제거율: 54.2%)
상위권 - 전처리 전: 5897, 후: 2785 (제거율: 52.8%)

=== 순수 범주형 감성 데이터 집계 ===

하위권 뉴스 데이터 전처리:
  원본 데

### 가설2 TABLE2 (상관관계행렬) 생성

In [ ]:
pd.DataFrame(final_df[['ind_IT_SW_통신','ind_바이오_제약','ind_제조업','ind_금융_투자','sentiment_mean_scaled','info_group','log_news_count_scaled',
                                                'log_assets_scaled','leverage_scaled','MBV','log_MBV']].corr(numeric_only=True))

,ind_IT_SW_통신,ind_바이오_제약,ind_제조업,ind_금융_투자,sentiment_mean_scaled,info_group,log_news_count_scaled,log_assets_scaled,leverage_scaled,MBV,log_MBV
ind_IT_SW_통신,1.000000,-0.337187,-0.644076,-0.262235,-0.190601,-0.077378,0.373703,-0.276370,-0.320376,-0.059507,-0.014655
ind_바이오_제약,-0.337187,1.000000,-0.169791,-0.069130,-0.017738,-0.036784,-0.098619,-0.174541,0.017762,0.121507,0.124282
ind_제조업,-0.644076,-0.169791,1.000000,-0.132048,0.134325,0.025373,-0.319091,0.238959,0.362213,0.057248,0.012913
ind_금융_투자,-0.262235,-0.069130,-0.132048,1.000000,0.089142,0.205020,-0.150703,0.211219,0.096081,-0.022775,-0.039705
sentiment_mean_scaled,-0.190601,-0.017738,0.134325,0.089142,1.000000,-0.249487,-0.128938,0.395163,-0.095175,-0.080817,-0.214941
info_group,-0.077378,-0.036784,0.025373,0.205020,-0.249487,1.000000,0.183080,-0.390193,-0.059272,0.406353,0.474093
log_news_count_scaled,0.373703,-0.098619,-0.319091,-0.150703,-0.128938,0.183080,1.000000,-0.277575,-0.243202,0.237916,0.281344
log_assets_scaled,-0.276370,-0.174541,0.238959,0.211219,0.395163,-0.390193,-0.277575,1.000000,0.134779,-0.307149,-0.507904
leverage_scaled,-0.320376,0.017762,0.362213,0.096081,-0.095175,-0.059272,-0.243202,0.134779,1.000000,0.166276,0.081124
MBV,-0.059507,0.121507,0.057248,-0.022775,-0.080817,0.406353,0.237916,-0.307149,0.166276,1.000000,0.879040


In [ ]:
pd.DataFrame(final_df[['ind_IT_SW_통신','ind_바이오_제약','ind_제조업','ind_금융_투자','sentiment_mean_scaled','info_group','log_news_count_scaled',
                                                'log_assets_scaled','leverage_scaled','MBV','log_MBV']].corr(numeric_only=True)).to_csv("TABLE2_가설2_최종검정때변수_상관관계행렬.csv",encoding='utf-8-sig')

## 가설1 검정


In [ ]:
print("뉴스 감성 데이터 로딩 중...")
news_df = pd.read_csv('/content/뉴스감성분석(개선판_키워드3개기사만)_20250922_021928.csv', encoding='utf-8-sig')
print(f"✓ 뉴스 감성 데이터: {news_df.shape}")

# 1-2. 재무 데이터
print("재무 데이터 로딩 중...")
financial_df = pd.read_excel('/content/dart_financial_merged(약700개_10년치)재무재표정보).xlsx')
print(f"✓ 재무 데이터: {financial_df.shape}")

# 1-3. 시가총액 데이터
print("시가총액 데이터 로딩 중...")
market_df = pd.read_csv('/content/market_table_quarterly(KRX시가총액정보_상장주식수_dart기준기업).csv', encoding='utf-8-sig')
print(f"✓ 시가총액 데이터: {market_df.shape}")

뉴스 감성 데이터 로딩 중...
✓ 뉴스 감성 데이터: (12789, 18)
재무 데이터 로딩 중...
✓ 재무 데이터: (18701, 11)
시가총액 데이터 로딩 중...
✓ 시가총액 데이터: (21892, 9)


In [ ]:
print(news_df.info(),
      news_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12789 entries, 0 to 12788
Data columns (total 18 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   종목명      12789 non-null  object 
 1   종목코드     12789 non-null  object 
 2   기사날짜     12789 non-null  object 
 3   기사제목     12789 non-null  object 
 4   기사링크     12789 non-null  object 
 5   본문요약     12789 non-null  object 
 6   긍정확률     12789 non-null  float64
 7   부정확률     12789 non-null  float64
 8   신뢰도      12789 non-null  float64
 9   감성점수     12789 non-null  float64
 10  탐지된기업    12789 non-null  object 
 11  탐지된종목코드  12789 non-null  object 
 12  키워드소스    12789 non-null  object 
 13  분석버전     12789 non-null  object 
 14  GPU디바이스  12789 non-null  object 
 15  처리시간     12789 non-null  float64
 16  혼합정밀도    12789 non-null  object 
 17  분석일시     12789 non-null  object 
dtypes: float64(5), object(13)
memory usage: 1.8+ MB
None                긍정확률          부정확률           신뢰도          감성점수          처리

In [ ]:
print(financial_df.info(),
      financial_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18701 entries, 0 to 18700
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   자산총계         18686 non-null  float64
 1   부채총계         18699 non-null  float64
 2   자본총계         18699 non-null  float64
 3   매출액          18260 non-null  float64
 4   당기순이익        18683 non-null  float64
 5   bsns_year    18701 non-null  int64  
 6   reprt_code   18701 non-null  int64  
 7   report_name  18701 non-null  object 
 8   report_date  18701 non-null  object 
 9   기업명          18701 non-null  object 
 10  종목코드         18701 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 1.6+ MB
None                자산총계          부채총계          자본총계           매출액         당기순이익  \
count  1.868600e+04  1.869900e+04  1.869900e+04  1.826000e+04  1.868300e+04   
mean   1.558270e+13  7.069916e+12  8.500228e+12  4.815556e+12 -1.011682e+12   
std    1.150334e+15  4.827120e+14  6.6901

In [ ]:
print(market_df.info(),
      market_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21892 entries, 0 to 21891
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   market_cap_krw      21892 non-null  int64  
 1   shares_outstanding  21892 non-null  int64  
 2   volume              21892 non-null  int64  
 3   value_traded        21892 non-null  int64  
 4   close_price         7830 non-null   float64
 5   as_of_date          21892 non-null  int64  
 6   period_yyyyq        21892 non-null  object 
 7   ticker              21892 non-null  int64  
 8   firm_id             21892 non-null  int64  
dtypes: float64(1), int64(7), object(1)
memory usage: 1.5+ MB
None        market_cap_krw  shares_outstanding        volume  value_traded  \
count    2.189200e+04        2.189200e+04  2.189200e+04  2.189200e+04   
mean     9.547161e+11        5.224348e+07  4.066473e+05  4.999980e+09   
std      1.127498e+13        2.311575e+08  6.167797e+06  3.922283e+

### 가설1 강건한, 강건성 범주형감성점수 테스트 (이게 찐찐찐 찐가설1 마지막임)

In [6]:
pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.1 MB/s eta 0:00:00


In [9]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import RobustScaler
from scipy.stats.mstats import winsorize
import warnings

warnings.filterwarnings("ignore")

# ======================================================
# 공통: 유의성 표시
# ======================================================

def signif(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""


# ======================================================
# 1. 가설1 데이터 로딩
# ======================================================

def load_h1_data():
    news_df = pd.read_csv('/content/최종검정용뉴스감성_더미감성(전체매칭_가설1).csv', encoding='utf-8-sig')
    financial_df = pd.read_excel('/content/dart_financial_merged(약700개_10년치)재무재표정보).xlsx')
    market_df = pd.read_csv('/content/market_table_quarterly(KRX시가총액정보_상장주식수_dart기준기업).csv', encoding='utf-8-sig')
    return news_df, financial_df, market_df


# ======================================================
# 2. 정제
# ======================================================

def clean_and_standardize_data(news_df, financial_df, market_df):

    # --- 뉴스 ---
    news = news_df.copy()
    news['종목코드'] = pd.to_numeric(news['종목코드'], errors='coerce')
    news = news.dropna(subset=['종목코드'])
    news['종목코드'] = news['종목코드'].astype(int)

    news['기사날짜'] = pd.to_datetime(news['기사날짜'], errors='coerce')
    news = news.dropna(subset=['기사날짜'])
    news['year'] = news['기사날짜'].dt.year

    news = news.dropna(subset=['감성점수_범주형'])
    news = news[news['감성점수_범주형'].isin([1,2,3])]

    # --- 재무 ---
    fin = financial_df.copy()
    fin['종목코드'] = fin['종목코드'].astype(int)
    fin['year'] = fin['bsns_year']

    fin = fin[
        (fin['자산총계']>0) &
        (fin['자본총계']>0) &
        (fin['부채총계']>=0) &
        (fin['당기순이익'].notna())
    ]

    for col in ['자산총계','자본총계','부채총계','매출액']:
        q01 = fin[col].quantile(0.01)
        q99 = fin[col].quantile(0.99)
        fin = fin[(fin[col] >= q01) & (fin[col] <= q99)]

    # --- 시가총액 ---
    market = market_df.copy()
    market['종목코드'] = market['ticker'].astype(int)
    market['year'] = market['period_yyyyq'].str[:4].astype(int)

    market = market[
        (market['market_cap_krw']>0) &
        (market['shares_outstanding']>0)
    ]

    market_year = market.groupby(['종목코드','year']).agg({
        'market_cap_krw':'mean',
        'shares_outstanding':'mean',
        'volume':'mean',
        'value_traded':'mean'
    }).reset_index()

    return news, fin, market_year


# ======================================================
# 3. 범주형 감성 집계 (firm-year)
# ======================================================

def create_categorical_sentiment_aggregates(news):

    rows=[]
    for (tic,year), g in news.groupby(['종목코드','year']):
        if len(g) < 3:
            continue

        total=len(g)
        neg=(g['감성점수_범주형']==1).sum()
        neu=(g['감성점수_범주형']==2).sum()
        pos=(g['감성점수_범주형']==3).sum()

        rows.append({
            'firm_code': tic,
            'year': year,
            'sentiment_mean': g['감성점수_범주형'].mean(),
            'sentiment_normalized': (g['감성점수_범주형'].mean() - 2),
            'news_count': total,
            'positive_ratio': pos/total,
            'neutral_ratio': neu/total,
            'negative_ratio': neg/total
        })

    return pd.DataFrame(rows)


# ======================================================
# 4. 재무 비율 생성 (영문 피처명 유지)
# ======================================================

def create_financial_ratios(fin):

    f = fin.copy()
    f['firm_code'] = f['종목코드']
    f['debt_to_equity'] = f['부채총계'] / f['자본총계']
    f['roa'] = f['당기순이익'] / f['자산총계']
    f['log_assets'] = np.log(f['자산총계'])

    out=[]
    for tic, g in f.groupby('firm_code'):
        g = g.sort_values('year')
        g['asset_growth'] = g['자산총계'].pct_change()
        out.append(g)
    return pd.concat(out, ignore_index=True)


# ======================================================
# 5. 병합 & MBV 계산
# ======================================================

def merge_all_data(sentiment, fin, market):

    fin2 = fin.copy()
    fin2['firm_code'] = fin2['종목코드']

    market2 = market.copy()
    market2['firm_code'] = market2['종목코드']

    df = pd.merge(fin2, market2, on=['firm_code','year'], how='inner')
    df = pd.merge(df, sentiment, on=['firm_code','year'], how='inner')

    df['MBV'] = df['market_cap_krw'] / df['자본총계']
    df['MBV_winsor'] = winsorize(df['MBV'], limits=[0.02,0.02])
    df['market_liquidity'] = np.log(df['value_traded']+1)

    df = df.sort_values(['firm_code','year'])
    df['obs_per_firm'] = df.groupby('firm_code')['MBV'].transform('count')
    df = df[df['obs_per_firm']>=3]

    return df


# ======================================================
# 6. 스케일링 + 연도 더미
# ======================================================

def prepare_panel_for_h1(df):

    scaler = RobustScaler()
    for col in ['sentiment_mean','log_assets','debt_to_equity','roa','market_liquidity']:
        if col in df.columns:
            df[col + '_scaled'] = scaler.fit_transform(df[[col]])

    freq_year = df['year'].value_counts().idxmax()
    year_dummies = []
    for y in sorted(df['year'].unique()):
        if y != freq_year:
            col = f'year_{y}'
            df[col] = (df['year'] == y).astype(int)
            year_dummies.append(col)

    return df, year_dummies


# ======================================================
# 7. 한 샘플에 대해 cluster OLS → 정석 회귀표 DataFrame 만들기
# ======================================================

def run_cluster_ols_table(df_group, year_dummies):

    dep_var = 'MBV_winsor'
    key_var = 'sentiment_mean_scaled'
    controls = ['log_assets_scaled', 'debt_to_equity_scaled', 'roa_scaled', 'market_liquidity_scaled'] + year_dummies

    cols_needed = [dep_var, key_var] + controls + ['firm_code']
    d = df_group.dropna(subset=cols_needed).copy()

    if len(d) < 40:
        # 표본 너무 작으면 빈 테이블 리턴
        return pd.DataFrame(columns=[
            "변수","계수","표준오차","z값","p-value",
            "하한 95% CI","상한 95% CI","유의성","계수(유의성)","(표준오차)"
        ])

    X = d[[key_var] + controls]
    X = sm.add_constant(X)
    y = d[dep_var]

    model = sm.OLS(y, X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d['firm_code']})

    params = res.params
    bse    = res.bse
    pvals  = res.pvalues
    conf   = res.conf_int(alpha=0.05)

    # “const” 포함 전체 변수별 행 생성
    rows=[]
    for var in params.index:
        coef = params[var]
        se   = bse[var]
        p    = pvals[var]
        z    = coef / se if se != 0 else np.nan
        ci_low, ci_high = conf.loc[var]

        star = signif(p)
        coef_star = f"{coef:.4f}{star}"
        se_str    = f"({se:.4f})"

        # 변수명은 영어 그대로 두고, 필요하면 엑셀에서 네가 label 손 봐
        rows.append({
            "변수": var,
            "계수": coef,
            "표준오차": se,
            "z값": z,
            "p-value": p,
            "하한 95% CI": ci_low,
            "상한 95% CI": ci_high,
            "유의성": star,
            "계수(유의성)": coef_star,
            "(표준오차)": se_str
        })

    table = pd.DataFrame(rows)
    return table


# ======================================================
# 8. 샘플별 시트 1개씩 생성해서 엑셀 저장
#    Full / Large / HighGrowth / Recent3Y / HighNews
# ======================================================

def save_h1_multi_sheets(df, year_dummies,
                         path="가설검정_결과정리_범주형감성(H1결과_정석표).xlsx"):

    # 샘플 정의
    full = df.copy()
    large = df[df['log_assets'] >= df['log_assets'].median()].copy()
    if 'asset_growth' in df.columns:
        highg = df[df['asset_growth'] >= 0].copy()
    else:
        highg = df.iloc[0:0].copy()
    recent3 = df[df['year'] >= df['year'].max() - 2].copy()
    highnews = df[df['news_count'] >= 10].copy()

    samples = [
        ("Full sample", full),
        ("Large firms (log_assets ≥ median)", large),
        ("High growth (asset_growth ≥ 0)", highg),
        ("Recent 3 years", recent3),
        ("High news coverage (news_count ≥ 10)", highnews)
    ]

    with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        for sheet_name, dsub in samples:
            table = run_cluster_ols_table(dsub, year_dummies)
            # 시트 이름은 엑셀 제한 때문에 좀 줄여줌
            safe_sheet = sheet_name
            if len(safe_sheet) > 31:
                safe_sheet = safe_sheet[:31]
            if table.empty:
                # 표본 작을 때도 틀은 만들기
                table = pd.DataFrame(columns=[
                    "변수","계수","표준오차","z값","p-value",
                    "하한 95% CI","상한 95% CI","유의성","계수(유의성)","(표준오차)"
                ])
            table.to_excel(writer, sheet_name=safe_sheet, index=False)

    print(f"\n✅ H1 엑셀 저장 완료 (샘플별 시트 1개씩): {path}")


# ======================================================
# 9. 메인 실행
# ======================================================

def main_h1():
    news_df, fin_df, market_df = load_h1_data()
    news_clean, fin_clean, market_year = clean_and_standardize_data(news_df, fin_df, market_df)

    sentiment_agg = create_categorical_sentiment_aggregates(news_clean)
    fin_feat = create_financial_ratios(fin_clean)

    final_df = merge_all_data(sentiment_agg, fin_feat, market_year)
    final_df, year_dummies = prepare_panel_for_h1(final_df)

    save_h1_multi_sheets(final_df, year_dummies)

# 실행
if __name__ == "__main__":
    main_h1()



✅ H1 엑셀 저장 완료 (샘플별 시트 1개씩): 가설검정_결과정리_범주형감성(H1결과_정석표).xlsx


### 가설용 테이블 작성

#### 실제 HTML과 그래프로 표 생성

In [ ]:
#### 구버전 코드

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.stats import pearsonr, spearmanr, zscore, jarque_bera
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.sandwich_covariance import cov_hac
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# 새로 추가된 라이브러리들 (논문용 표 시각화)
import matplotlib.patches as patches
from matplotlib.table import Table
import matplotlib.font_manager as fm
from io import BytesIO
import base64

warnings.filterwarnings('ignore')

# ===============================
# 학술표준 논문용 표 생성 함수들
# ===============================

def stars(p):
    """유의성 표시"""
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ('†' if p < 0.10 else '')))

def format_coef_with_pvalue(coef, se, pval):
    """계수와 p값을 학술표준 형식으로 포매팅"""
    star = stars(pval)
    return f"{coef:.3f}{star}\n({se:.3f})\n[{pval:.3f}]"

def extract_table_block_enhanced(res, var_order=None, var_rename=None):
    """회귀분석 결과에서 계수/표준오차/p값 추출 (향상된 버전)"""
    params = res.params.copy()
    bse = res.bse.copy()
    pvals = res.pvalues.copy()

    if var_order is None:
        var_order = [v for v in params.index if v.lower() not in ('const', '_cons', 'intercept')]

    def pretty_name(name):
        if var_rename and name in var_rename:
            return var_rename[name]
        return name

    rows = {}
    for v in var_order:
        if v in params.index:
            coef = params[v]
            se = bse[v]
            pv = pvals[v]
            rows[pretty_name(v)] = format_coef_with_pvalue(coef, se, pv)
        else:
            rows[pretty_name(v)] = ""

    return pd.Series(rows, name="")

def make_regression_table_enhanced(results_list, model_names=None, var_order=None, var_rename=None, add_rows=None, show_adj_r2=True):
    """학술표준 논문용 회귀분석 표 생성"""

    if model_names is None:
        model_names = [f"({i+1})" for i in range(len(results_list))]

    # 계수 블록 생성
    cols = []
    for res in results_list:
        col = extract_table_block_enhanced(res, var_order=var_order, var_rename=var_rename)
        cols.append(col)

    body = pd.concat(cols, axis=1)
    body.columns = model_names

    # 하단 통계 정보
    foot_rows = []

    # 관측치 수
    n_list = [int(res.nobs) for res in results_list]
    foot_rows.append(pd.Series({m: f"{n:,d}" for m, n in zip(model_names, n_list)}, name="Observations"))

    # R-squared
    if show_adj_r2:
        adj_r2_list = [getattr(res, "rsquared_adj", np.nan) for res in results_list]
        foot_rows.append(pd.Series({m: f"{r:.3f}" for m, r in zip(model_names, adj_r2_list)}, name="Adj. R-squared"))
    else:
        r2_list = [res.rsquared for res in results_list]
        foot_rows.append(pd.Series({m: f"{r:.3f}" for m, r in zip(model_names, r2_list)}, name="R-squared"))

    # F-통계량
    f_list = [getattr(res, "fvalue", np.nan) for res in results_list]
    foot_rows.append(pd.Series({m: f"{f:.2f}" for m, f in zip(model_names, f_list)}, name="F-statistic"))

    # 사용자 정의 메타정보
    if add_rows:
        for k, v in add_rows.items():
            if isinstance(v, list):
                ser = pd.Series({m: v[i] for i, m in enumerate(model_names)}, name=k)
            else:
                ser = pd.Series({m: v for m in model_names}, name=k)
            foot_rows.append(ser)

    # 전체 표 결합
    foot_df = pd.DataFrame(foot_rows)
    table = pd.concat([body, foot_df], axis=0)

    return table

def create_academic_standard_html(table, title="Regression Results"):
    """APA/AER 표준을 따르는 학술논문용 HTML 표 생성"""

    html_templates = []
    font_sizes = ['11px', '12px', '13px']
    size_labels = ['Small', 'Medium', 'Large']

    for i, (font_size, size_label) in enumerate(zip(font_sizes, size_labels)):

        html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title} - {size_label}</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Times+New+Roman:wght@400;700&display=swap');

        body {{
            font-family: "Times New Roman", Times, serif;
            margin: 30px;
            background-color: white;
            line-height: 1.2;
        }}

        .table-container {{
            max-width: 900px;
            margin: 0 auto;
            background-color: white;
            padding: 20px;
        }}

        .table-title {{
            text-align: center;
            font-size: {str(int(font_size.replace('px', '')) + 3)}px;
            font-weight: bold;
            margin-bottom: 20px;
            color: #000;
            font-family: "Times New Roman", Times, serif;
        }}

        .regression-table {{
            width: 100%;
            border-collapse: collapse;
            margin: 0 auto;
            font-size: {font_size};
            background-color: white;
            font-family: "Times New Roman", Times, serif;
        }}

        .regression-table th {{
            background-color: white;
            border-top: 2px solid #000;
            border-bottom: 1px solid #000;
            border-left: none;
            border-right: none;
            padding: 8px 6px;
            text-align: center;
            font-weight: bold;
            color: #000;
            height: 40px;
            vertical-align: middle;
        }}

        .regression-table th:first-child {{
            text-align: left;
            padding-left: 0px;
        }}

        .regression-table td {{
            border: none;
            padding: 6px 6px;
            text-align: center;
            background-color: white;
            color: #000;
            height: 60px;
            vertical-align: middle;
            line-height: 1.1;
        }}

        .regression-table .var-name {{
            text-align: left;
            font-weight: normal;
            background-color: white;
            padding-left: 0px;
        }}

        .regression-table .stats-section {{
            border-top: 1px solid #000;
        }}

        .regression-table .stats-row td {{
            height: 30px;
            padding: 4px 6px;
        }}

        .regression-table .final-row {{
            border-bottom: 2px solid #000;
        }}

        .footnote {{
            font-size: {str(int(font_size.replace('px', '')) - 1)}px;
            margin-top: 15px;
            color: #000;
            line-height: 1.4;
            font-family: "Times New Roman", Times, serif;
        }}

        .coefficient {{
            font-weight: normal;
        }}

        .std-error {{
            font-style: italic;
            color: #333;
        }}

        .p-value {{
            font-size: {str(int(font_size.replace('px', '')) - 1)}px;
            color: #666;
        }}

        @media print {{
            body {{
                margin: 0;
            }}
            .table-container {{
                max-width: none;
            }}
        }}

        .interpretation-section {{
            margin-top: 30px;
            padding: 20px;
            background-color: #f9f9f9;
            border-left: 4px solid #333;
        }}

        .interpretation-title {{
            font-size: {str(int(font_size.replace('px', '')) + 1)}px;
            font-weight: bold;
            margin-bottom: 15px;
            color: #000;
        }}

        .interpretation-content {{
            font-size: {font_size};
            line-height: 1.6;
            color: #333;
        }}

        .indicator-explanation {{
            margin: 10px 0;
            padding: 8px;
            background-color: white;
            border-left: 2px solid #666;
        }}

        .indicator-name {{
            font-weight: bold;
            color: #000;
        }}
    </style>
</head>
<body>
    <div class="table-container">
        <div class="table-title">{title}</div>
        <table class="regression-table">
            <thead>
                <tr>
                    <th style="text-align: left;">Variable</th>
"""

        # 헤더 생성
        for col in table.columns:
            html += f'                    <th>{col.replace("\\n", "<br>")}</th>\n'

        html += """                </tr>
            </thead>
            <tbody>
"""

        # 데이터 행 생성
        stats_start_idx = None
        rows_list = list(table.iterrows())

        for idx, (row_name, row_data) in enumerate(rows_list):
            # 통계 섹션 시작점 감지
            if row_name in ['Observations', 'R-squared', 'Adj. R-squared', 'F-statistic'] and stats_start_idx is None:
                stats_start_idx = idx

            # 클래스 결정
            row_class = ''
            if stats_start_idx is not None and idx == stats_start_idx:
                row_class = 'stats-section'
            if stats_start_idx is not None and idx >= stats_start_idx:
                row_class += ' stats-row'
            if idx == len(rows_list) - 1:  # 마지막 행
                row_class += ' final-row'

            html += f'                <tr class="{row_class.strip()}">\n'
            html += f'                    <td class="var-name">{row_name}</td>\n'

            for col_name, cell_value in row_data.items():
                # 줄바꿈 처리
                formatted_value = str(cell_value)

                # 계수/표준오차/p값 형식 처리
                if '\n' in formatted_value and '[' in formatted_value:
                    parts = formatted_value.split('\n')
                    if len(parts) >= 3:
                        coef_part = f'<span class="coefficient">{parts[0]}</span>'
                        se_part = f'<span class="std-error">{parts[1]}</span>'
                        p_part = f'<span class="p-value">{parts[2]}</span>'
                        formatted_value = f"{coef_part}<br>{se_part}<br>{p_part}"
                    else:
                        formatted_value = formatted_value.replace('\n', '<br>')
                else:
                    formatted_value = formatted_value.replace('\n', '<br>')

                html += f'                    <td>{formatted_value}</td>\n'

            html += '                </tr>\n'

        html += """            </tbody>
        </table>

        <div class="footnote">
            <p><strong>Notes:</strong> Standard errors are reported in parentheses, p-values in brackets.
            *** p&lt;0.001, ** p&lt;0.01, * p&lt;0.05, † p&lt;0.10.
            Robust standard errors are used for all specifications.</p>
        </div>

        <!-- 해석 가이드 섹션 -->
        <div class="interpretation-section">
            <div class="interpretation-title">Statistical Indicators Interpretation Guide</div>
            <div class="interpretation-content">

                <div class="indicator-explanation">
                    <div class="indicator-name">News Sentiment:</div>
                    Measures the impact of news sentiment on firm valuation (Market-to-Book ratio).
                    Positive coefficient indicates that positive news sentiment increases firm value.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Firm Size (log):</div>
                    Natural logarithm of total assets. Controls for firm size effects.
                    Larger firms may have different valuation patterns.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Leverage:</div>
                    Debt-to-equity ratio. Higher leverage may indicate higher financial risk,
                    potentially affecting firm valuation.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">ROA (Return on Assets):</div>
                    Measures operational efficiency. Higher ROA typically indicates
                    better management performance and should positively affect valuation.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Market Liquidity:</div>
                    Trading volume measure. Higher liquidity may reduce information asymmetry
                    and affect the relationship between news sentiment and valuation.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Year Fixed Effects:</div>
                    Controls for time-specific factors affecting all firms in a given year
                    (e.g., macroeconomic conditions, regulatory changes).
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Adj. R-squared:</div>
                    Adjusted coefficient of determination. Indicates the proportion of variance
                    in the dependent variable explained by the model, adjusted for degrees of freedom.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">F-statistic:</div>
                    Tests the overall significance of the regression model.
                    Higher values indicate better model fit.
                </div>

                <div class="indicator-explanation">
                    <div class="indicator-name">Statistical Significance:</div>
                    *** p&lt;0.001 (highly significant), ** p&lt;0.01 (significant),
                    * p&lt;0.05 (significant), † p&lt;0.10 (marginal significance)
                </div>

            </div>
        </div>

    </div>
</body>
</html>"""

        html_templates.append((size_label, html))

    return html_templates

def create_academic_standard_image(table, title="Regression Results"):
    """학술표준을 따르는 논문용 이미지 표 생성"""

    # 폰트 설정 - 시스템에서 Times New Roman 계열 찾기
    font_name = 'serif'  # 기본값
    try:
        available_fonts = [f.name for f in fm.fontManager.ttflist]

        # 우선순위: Times New Roman 계열
        font_candidates = [
            'Times New Roman',
            'Times',
            'Liberation Serif',
            'DejaVu Serif',
            'serif'
        ]

        for font in font_candidates:
            if font in available_fonts:
                font_name = font
                break

        plt.rcParams['font.family'] = font_name
        plt.rcParams['font.serif'] = [font_name]

    except Exception as e:
        print(f"폰트 설정 경고: {e}")
        plt.rcParams['font.family'] = 'serif'

    image_files = []
    font_sizes = [9, 11, 13]
    figsize_list = [(12, 10), (14, 12), (16, 14)]
    size_labels = ['Small', 'Medium', 'Large']

    for i, (font_size, figsize, size_label) in enumerate(zip(font_sizes, figsize_list, size_labels)):

        fig, ax = plt.subplots(figsize=figsize, facecolor='white', dpi=300)
        ax.axis('off')

        # 제목
        fig.suptitle(title, fontsize=font_size+3, fontweight='bold', y=0.95,
                    fontfamily='serif')

        # 표 데이터 준비
        table_data = []
        headers = ['Variable'] + list(table.columns)

        # 데이터 행들
        for row_name, row_data in table.iterrows():
            row = [row_name]
            for col_value in row_data:
                # p값 포함 형식 처리
                formatted_value = str(col_value)
                if '\n' in formatted_value and '[' in formatted_value:
                    # 3줄 형식: 계수\n(표준오차)\n[p값]
                    parts = formatted_value.split('\n')
                    if len(parts) >= 3:
                        # 별표 분리
                        coef_line = parts[0]
                        se_line = parts[1]
                        p_line = parts[2]
                        formatted_value = f"{coef_line}\n{se_line}\n{p_line}"
                row.append(formatted_value)
            table_data.append(row)

        # 표 그리기
        n_rows = len(table_data) + 1  # +1 for header
        n_cols = len(headers)

        # 표 위치 및 크기 설정
        table_ax = fig.add_subplot(111)
        table_ax.axis('off')

        # 균등한 셀 크기
        table_width = 0.85
        table_height = 0.75
        cell_width = table_width / n_cols
        cell_height = table_height / n_rows

        start_x = 0.075
        start_y = 0.8

        # 헤더 그리기
        for j, header in enumerate(headers):
            x_pos = start_x + j * cell_width
            y_pos = start_y

            # 헤더 상단 굵은 선
            if j == 0:
                table_ax.plot([start_x, start_x + table_width], [y_pos + cell_height*0.1, y_pos + cell_height*0.1],
                             'k-', linewidth=2)

            # 헤더 텍스트
            ha = 'left' if j == 0 else 'center'
            table_ax.text(x_pos + (0.01 if j == 0 else cell_width/2), y_pos - cell_height/2,
                         header.replace('\\n', '\n'),
                         ha=ha, va='center', fontsize=font_size+1, fontweight='bold',
                         fontfamily='serif')

            # 헤더 하단 선
            if j == 0:
                table_ax.plot([start_x, start_x + table_width], [y_pos - cell_height*0.9, y_pos - cell_height*0.9],
                             'k-', linewidth=1)

        # 데이터 행 그리기
        stats_start_idx = None
        for i, row in enumerate(table_data):
            # 통계 섹션 감지
            if row[0] in ['Observations', 'R-squared', 'Adj. R-squared', 'F-statistic'] and stats_start_idx is None:
                stats_start_idx = i
                # 통계 섹션 시작 선
                y_line = start_y - (i + 1) * cell_height + cell_height*0.1
                table_ax.plot([start_x, start_x + table_width], [y_line, y_line],
                             'k-', linewidth=1)

            for j, cell_value in enumerate(row):
                x_pos = start_x + j * cell_width
                y_pos = start_y - (i + 1) * cell_height

                # 텍스트 정렬
                ha = 'left' if j == 0 else 'center'
                x_text = x_pos + (0.01 if j == 0 else cell_width/2)

                # 텍스트 크기 조정
                text_size = font_size if stats_start_idx is None or i < stats_start_idx else font_size - 1

                # 다줄 텍스트 처리
                if '\n' in str(cell_value) and j > 0:
                    lines = str(cell_value).split('\n')
                    line_spacing = cell_height / (len(lines) + 1)
                    for k, line in enumerate(lines):
                        line_y = y_pos - cell_height/2 + (len(lines)/2 - k) * line_spacing * 0.7

                        # 스타일 적용
                        weight = 'normal'
                        color = 'black'
                        if k == 1:  # 표준오차
                            color = '#444'
                        elif k == 2:  # p값
                            color = '#666'
                            text_size = font_size - 1

                        table_ax.text(x_text, line_y, line,
                                     ha=ha, va='center', fontsize=text_size,
                                     fontweight=weight, color=color, fontfamily='serif')
                else:
                    table_ax.text(x_text, y_pos - cell_height/2, str(cell_value),
                                 ha=ha, va='center', fontsize=text_size,
                                 fontweight='normal', fontfamily='serif')

        # 마지막 하단 굵은 선
        final_y = start_y - n_rows * cell_height + cell_height*0.1
        table_ax.plot([start_x, start_x + table_width], [final_y, final_y],
                     'k-', linewidth=2)

        # 범례
        footnote_y = final_y - 0.05
        footnote_text = ('Notes: Standard errors in parentheses, p-values in brackets. '
                        '*** p<0.001, ** p<0.01, * p<0.05, † p<0.10. '
                        'Robust standard errors used.')

        table_ax.text(start_x, footnote_y, footnote_text,
                     fontsize=font_size-1, ha='left', va='top',
                     fontfamily='serif', wrap=True)

        # 레이아웃 조정
        plt.tight_layout()
        plt.subplots_adjust(top=0.92, bottom=0.08)

        # 파일 저장
        filename = f'academic_table_{size_label.lower()}.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight',
                   facecolor='white', edgecolor='none', format='png',
                   pad_inches=0.2)

        image_files.append((size_label, filename))
        plt.close()

    return image_files

def save_academic_standard_formats(table, filename_base="academic_regression", title="Effect of News Sentiment on Market-to-Book Value"):
    """학술표준 형식으로 표 저장"""

    print(f"\n📊 학술표준 회귀분석 표:")
    print("="*100)
    print(table)
    print("="*100)

    # 1. 기존 형식들 저장
    try:
        excel_file = f"{filename_base}.xlsx"
        table.to_excel(excel_file, index=True)
        print(f"✓ Excel 저장: {excel_file}")
    except Exception as e:
        print(f"Excel 저장 실패: {e}")

    try:
        csv_file = f"{filename_base}.csv"
        table.to_csv(csv_file, index=True, encoding='utf-8-sig')
        print(f"✓ CSV 저장: {csv_file}")
    except Exception as e:
        print(f"CSV 저장 실패: {e}")

    # 2. 학술표준 HTML 저장
    print(f"\n🎓 학술표준 시각적 표 생성 중...")
    try:
        html_versions = create_academic_standard_html(table, title)

        for size_label, html_content in html_versions:
            html_file = f"{filename_base}_{size_label.lower()}.html"
            with open(html_file, 'w', encoding='utf-8') as f:
                f.write(html_content)
            print(f"✓ 학술표준 HTML ({size_label}): {html_file}")

    except Exception as e:
        print(f"HTML 생성 실패: {e}")

    # 3. 학술표준 이미지 저장
    try:
        image_versions = create_academic_standard_image(table, title)

        for size_label, image_file in image_versions:
            print(f"✓ 학술표준 이미지 ({size_label}): {image_file}")

    except Exception as e:
        print(f"이미지 생성 실패: {e}")

    print(f"\n🏆 학술표준 논문 표 완성!")
    print(f"   📋 특징:")
    print(f"      • APA/AER 저널 스타일 준수")
    print(f"      • 계수 + 표준오차 + p값 모두 표시")
    print(f"      • 균등한 셀 높이")
    print(f"      • Times New Roman 폰트")
    print(f"      • 학술논문 표준 격자선")
    print(f"      • 통계 지표 해석 가이드 포함")
    print(f"   📐 크기: Small, Medium, Large")
    print(f"   📄 형식: HTML (스크린샷용) + PNG (직접삽입용)")

# ===============================
# 테스트용 Mock 데이터 생성
# ===============================

def create_mock_regression_results():
    """Mock 회귀분석 결과 생성"""

    # Mock 데이터 생성
    np.random.seed(42)
    n = 396

    data = pd.DataFrame({
        'MBV': np.random.normal(2.2, 0.8, n),
        'sentiment_mean_scaled': np.random.normal(0, 1, n),
        'log_assets_scaled': np.random.normal(0, 1, n),
        'debt_to_equity_scaled': np.random.normal(0, 1, n),
        'roa_scaled': np.random.normal(0, 1, n),
        'market_liquidity_scaled': np.random.normal(0, 1, n),
        'year_2021': np.random.binomial(1, 0.2, n),
        'year_2022': np.random.binomial(1, 0.2, n),
        'year_2023': np.random.binomial(1, 0.2, n)
    })

    # 회귀모델들 실행
    models = []
    model_names = []

    # 모델 1: 기본
    model1 = smf.ols('MBV ~ sentiment_mean_scaled', data=data).fit(cov_type='HC1')
    models.append(model1)
    model_names.append('Basic')

    # 모델 2: 기업통제
    model2 = smf.ols('MBV ~ sentiment_mean_scaled + log_assets_scaled + debt_to_equity_scaled + roa_scaled', data=data).fit(cov_type='HC1')
    models.append(model2)
    model_names.append('Firm Controls')

    # 모델 3: 시장통제
    model3 = smf.ols('MBV ~ sentiment_mean_scaled + log_assets_scaled + debt_to_equity_scaled + roa_scaled + market_liquidity_scaled', data=data).fit(cov_type='HC1')
    models.append(model3)
    model_names.append('Market Controls')

    # 모델 4: 시간통제
    model4 = smf.ols('MBV ~ sentiment_mean_scaled + log_assets_scaled + debt_to_equity_scaled + roa_scaled + market_liquidity_scaled + year_2021 + year_2022 + year_2023', data=data).fit(cov_type='HC1')
    models.append(model4)
    model_names.append('Year FE')

    return models, model_names

def test_academic_standard_table():
    """학술표준 표 생성 테스트"""

    print("🎓 학술표준 논문 표 생성 테스트")
    print("=" * 60)

    # Mock 데이터 생성
    models, model_names = create_mock_regression_results()

    # 변수 설정
    var_order = [
        'sentiment_mean_scaled',
        'log_assets_scaled',
        'debt_to_equity_scaled',
        'roa_scaled',
        'market_liquidity_scaled',
        'year_2021',
        'year_2022',
        'year_2023'
    ]

    var_rename = {
        'sentiment_mean_scaled': 'News Sentiment',
        'log_assets_scaled': 'Firm Size (log)',
        'debt_to_equity_scaled': 'Leverage',
        'roa_scaled': 'ROA',
        'market_liquidity_scaled': 'Market Liquidity',
        'year_2021': 'Year 2021',
        'year_2022': 'Year 2022',
        'year_2023': 'Year 2023'
    }

    # 모델명
    paper_model_names = []
    for i, name in enumerate(model_names):
        paper_model_names.append(f"({i+1})")

    # 추가 정보
    n_models = len(models)
    add_rows = {
        "Firm Controls": ["No"] + ["Yes"] * (n_models - 1),
        "Market Controls": ["No"] * 2 + ["Yes"] * (n_models - 2),
        "Year Fixed Effects": ["No"] * 3 + ["Yes"] * (n_models - 3),
        "Standard Errors": ["Robust"] * n_models
    }

    # 학술표준 표 생성
    table = make_regression_table_enhanced(
        results_list=models,
        model_names=paper_model_names,
        var_order=var_order,
        var_rename=var_rename,
        add_rows=add_rows,
        show_adj_r2=True
    )

    print("✅ 학술표준 회귀분석 표 생성 완료")

    # 학술표준 형식으로 저장
    save_academic_standard_formats(table, "academic_regression", "Effect of News Sentiment on Market-to-Book Value")

    print("\n🎯 학술표준 테스트 완료!")
    print("생성된 파일들:")
    print("   📄 HTML: academic_regression_small.html, medium.html, large.html")
    print("   🖼️ PNG: academic_table_small.png, medium.png, large.png")
    print("   📊 DATA: academic_regression.xlsx, .csv")

if __name__ == "__main__":
    test_academic_standard_table()

🎓 학술표준 논문 표 생성 테스트
✅ 학술표준 회귀분석 표 생성 완료

📊 학술표준 회귀분석 표:
                                        (1)                       (2)  \
News Sentiment      0.018\n(0.039)\n[0.649]   0.018\n(0.040)\n[0.646]   
Firm Size (log)                               0.020\n(0.037)\n[0.592]   
Leverage                                     -0.003\n(0.039)\n[0.934]   
ROA                                          -0.008\n(0.042)\n[0.843]   
Market Liquidity                                                        
Year 2021                                                               
Year 2022                                                               
Year 2023                                                               
Observations                            396                       396   
Adj. R-squared                       -0.002                    -0.009   
F-statistic                            0.21                      0.14   
Firm Controls                            No                       Yes

### 가설1 TABLE2 (상관관계행렬) 생성

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.stats import pearsonr, spearmanr, zscore, jarque_bera
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.sandwich_covariance import cov_hac
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ===============================
# 논문용 표 생성 함수들
# ===============================

def stars(p):
    """유의성 표시"""
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else ''))

def extract_table_block(res, var_order=None, var_rename=None):
    """회귀분석 결과에서 계수/표준오차/유의확률 추출"""
    params = res.params.copy()
    bse = res.bse.copy()
    pvals = res.pvalues.copy()

    if var_order is None:
        var_order = [v for v in params.index if v.lower() not in ('const', '_cons', 'intercept')]

    def pretty_name(name):
        if var_rename and name in var_rename:
            return var_rename[name]
        return name

    rows = {}
    for v in var_order:
        if v in params.index:
            coef = params[v]
            se = bse[v]
            pv = pvals[v]
            rows[pretty_name(v)] = f"{coef:.3f}{stars(pv)}\n({se:.3f})"
        else:
            rows[pretty_name(v)] = ""

    return pd.Series(rows, name="")

def make_regression_table(results_list, model_names=None, var_order=None, var_rename=None, add_rows=None, show_adj_r2=True):
    """논문용 회귀분석 표 생성"""

    if model_names is None:
        model_names = [f"({i+1})" for i in range(len(results_list))]

    # 계수 블록 생성
    cols = []
    for res in results_list:
        col = extract_table_block(res, var_order=var_order, var_rename=var_rename)
        cols.append(col)

    body = pd.concat(cols, axis=1)
    body.columns = model_names

    # 하단 통계 정보
    foot_rows = []

    # 관측치 수
    n_list = [int(res.nobs) for res in results_list]
    foot_rows.append(pd.Series({m: f"{n:,d}" for m, n in zip(model_names, n_list)}, name="Observations"))

    # R-squared
    if show_adj_r2:
        adj_r2_list = [getattr(res, "rsquared_adj", np.nan) for res in results_list]
        foot_rows.append(pd.Series({m: f"{r:.3f}" for m, r in zip(model_names, adj_r2_list)}, name="Adj. R-squared"))
    else:
        r2_list = [res.rsquared for res in results_list]
        foot_rows.append(pd.Series({m: f"{r:.3f}" for m, r in zip(model_names, r2_list)}, name="R-squared"))

    # F-통계량
    f_list = [getattr(res, "fvalue", np.nan) for res in results_list]
    foot_rows.append(pd.Series({m: f"{f:.2f}" for m, f in zip(model_names, f_list)}, name="F-statistic"))

    # 사용자 정의 메타정보
    if add_rows:
        for k, v in add_rows.items():
            if isinstance(v, list):
                ser = pd.Series({m: v[i] for i, m in enumerate(model_names)}, name=k)
            else:
                ser = pd.Series({m: v for m in model_names}, name=k)
            foot_rows.append(ser)

    # 전체 표 결합
    foot_df = pd.DataFrame(foot_rows)
    table = pd.concat([body, foot_df], axis=0)

    return table

def save_all_formats(table, filename_base="sentiment_mbv_results"):
    """표를 모든 형식으로 저장"""

    print(f"\n논문용 회귀분석 표:")
    print("="*100)
    print(table)
    print("="*100)

    # Excel 저장
    try:
        excel_file = f"{filename_base}.xlsx"
        table.to_excel(excel_file, index=True)
        print(f"✓ Excel 저장: {excel_file}")
    except Exception as e:
        print(f"Excel 저장 실패: {e}")

    # CSV 저장
    try:
        csv_file = f"{filename_base}.csv"
        table.to_csv(csv_file, index=True, encoding='utf-8-sig')
        print(f"✓ CSV 저장: {csv_file}")
    except Exception as e:
        print(f"CSV 저장 실패: {e}")

    # LaTeX 저장
    try:
        latex_code = table.to_latex(
            escape=False,
            column_format="l" + "c" * len(table.columns),
            caption="Effect of News Sentiment on Market-to-Book Value",
            label="tab:sentiment_mbv"
        )

        latex_file = f"{filename_base}.tex"
        with open(latex_file, 'w', encoding='utf-8') as f:
            f.write(latex_code)
        print(f"✓ LaTeX 저장: {latex_file}")

        print(f"\nLaTeX 코드:")
        print("-" * 60)
        print(latex_code)

    except Exception as e:
        print(f"LaTeX 저장 실패: {e}")

# ===============================
# 수정된 분석 함수들 (결과 반환)
# ===============================

def load_and_explore_data():
    """데이터 로딩 및 기초 탐색"""

    print("\n1. 데이터 로딩 및 기초 탐색")
    print("-" * 80)

    # 1-1. 뉴스 감성 데이터
    news_df = pd.read_csv('/content/뉴스감성분석(개선판_키워드3개기사만)_20250922_021928.csv', encoding='utf-8-sig')
    print(f"뉴스 데이터: {news_df.shape}")

    # 1-2. 재무 데이터
    financial_df = pd.read_excel('/content/dart_financial_merged(약700개_10년치)재무재표정보).xlsx')
    print(f"재무 데이터: {financial_df.shape}")

    # 1-3. 시가총액 데이터
    market_df = pd.read_csv('/content/market_table_quarterly(KRX시가총액정보_상장주식수_dart기준기업).csv', encoding='utf-8-sig')
    print(f"시가총액 데이터: {market_df.shape}")

    # 기초 통계 확인
    print(f"\n감성점수 분포:")
    print(f"평균: {news_df['감성점수'].mean():.4f}, 표준편차: {news_df['감성점수'].std():.4f}")
    print(f"범위: [{news_df['감성점수'].min():.3f}, {news_df['감성점수'].max():.3f}]")

    return news_df, financial_df, market_df

def clean_and_standardize_data(news_df, financial_df, market_df):
    """데이터 정제 및 표준화"""

    print("\n2. 데이터 정제 및 표준화")
    print("-" * 80)

    # 2-1. 뉴스 데이터 정제
    print("2-1. 뉴스 데이터 정제")
    news_clean = news_df.copy()

    # 종목코드 표준화 (6자리 숫자로 통일)
    news_clean['종목코드'] = pd.to_numeric(news_clean['종목코드'], errors='coerce')
    news_clean = news_clean.dropna(subset=['종목코드'])
    news_clean['종목코드'] = news_clean['종목코드'].astype(int)

    # 날짜 처리
    news_clean['기사날짜'] = pd.to_datetime(news_clean['기사날짜'], errors='coerce')
    news_clean = news_clean.dropna(subset=['기사날짜'])
    news_clean['year'] = news_clean['기사날짜'].dt.year
    news_clean['quarter'] = news_clean['기사날짜'].dt.quarter

    # 감성 데이터 이상치 제거 (표준 3시그마 규칙)
    for col in ['감성점수', '긍정확률', '부정확률', '신뢰도']:
        mean_val = news_clean[col].mean()
        std_val = news_clean[col].std()
        news_clean = news_clean[
            (news_clean[col] >= mean_val - 3*std_val) &
            (news_clean[col] <= mean_val + 3*std_val)
        ]

    print(f"   정제 후: {news_clean.shape}")

    # 2-2. 재무 데이터 정제
    print("2-2. 재무 데이터 정제")
    financial_clean = financial_df.copy()

    # 종목코드 표준화
    financial_clean['종목코드'] = financial_clean['종목코드'].astype(int)
    financial_clean['year'] = financial_clean['bsns_year']

    # 재무 데이터 필터링 (기본 조건)
    financial_clean = financial_clean[
        (financial_clean['자산총계'] > 0) &
        (financial_clean['자본총계'] > 0) &
        (financial_clean['부채총계'] >= 0) &
        (financial_clean['당기순이익'].notna())
    ]

    # 극단 이상치 제거 (상하위 1%)
    for col in ['자산총계', '자본총계', '부채총계', '매출액']:
        if col in financial_clean.columns:
            q01 = financial_clean[col].quantile(0.01)
            q99 = financial_clean[col].quantile(0.99)
            financial_clean = financial_clean[
                (financial_clean[col] >= q01) & (financial_clean[col] <= q99)
            ]

    print(f"   정제 후: {financial_clean.shape}")

    # 2-3. 시가총액 데이터 정제
    print("2-3. 시가총액 데이터 정제")
    market_clean = market_df.copy()

    # 종목코드 표준화
    market_clean['종목코드'] = market_clean['ticker'].astype(int)

    # 연도 추출
    market_clean['year'] = market_clean['period_yyyyq'].str[:4].astype(int)
    market_clean['quarter'] = market_clean['period_yyyyq'].str[-1:].astype(int)

    # 시가총액 유효성 체크
    market_clean = market_clean[
        (market_clean['market_cap_krw'] > 0) &
        (market_clean['shares_outstanding'] > 0)
    ]

    # 연도별 평균 계산 (분기 데이터 → 연간 데이터)
    market_yearly = market_clean.groupby(['종목코드', 'year']).agg({
        'market_cap_krw': 'mean',
        'shares_outstanding': 'mean',
        'volume': 'mean',
        'value_traded': 'mean'
    }).reset_index()

    print(f"   정제 후: {market_yearly.shape}")

    return news_clean, financial_clean, market_yearly

def create_sentiment_aggregates(news_df):
    """뉴스 감성 데이터 기업-연도별 집계"""

    print("\n3. 뉴스 감성 집계 (기업-연도별)")
    print("-" * 80)

    # 기업-연도별 집계 (최소 5개 기사 이상)
    sentiment_agg = news_df.groupby(['종목코드', 'year']).agg({
        '감성점수': ['mean', 'std', 'count', 'median'],
        '긍정확률': ['mean', 'std'],
        '부정확률': ['mean', 'std'],
        '신뢰도': ['mean', 'std']
    }).round(6)

    # 컬럼명 정리
    sentiment_agg.columns = [
        'sentiment_mean', 'sentiment_std', 'news_count', 'sentiment_median',
        'positive_mean', 'positive_std', 'negative_mean', 'negative_std',
        'confidence_mean', 'confidence_std'
    ]

    sentiment_agg = sentiment_agg.reset_index()

    # 뉴스 개수 필터링 (신뢰도 확보)
    min_news = 5
    sentiment_filtered = sentiment_agg[sentiment_agg['news_count'] >= min_news].copy()

    # 추가 지표 생성
    sentiment_filtered['sentiment_volatility'] = sentiment_filtered['sentiment_std'].fillna(0)
    sentiment_filtered['confidence_weighted_sentiment'] = (
        sentiment_filtered['sentiment_mean'] * sentiment_filtered['confidence_mean']
    )
    sentiment_filtered['news_intensity'] = sentiment_filtered['news_count'] / 365  # 일평균

    print(f"   집계 결과: {len(sentiment_agg)} → {len(sentiment_filtered)} (뉴스 {min_news}개 이상)")
    print(f"   평균 뉴스 개수: {sentiment_filtered['news_count'].mean():.1f}")

    return sentiment_filtered

def create_financial_ratios(financial_df):
    """재무 비율 및 지표 생성"""

    print("\n4. 재무 비율 및 지표 생성")
    print("-" * 80)

    financial_enhanced = financial_df.copy()

    # 기본 재무 비율
    financial_enhanced['debt_to_equity'] = financial_enhanced['부채총계'] / financial_enhanced['자본총계']
    financial_enhanced['debt_to_assets'] = financial_enhanced['부채총계'] / financial_enhanced['자산총계']
    financial_enhanced['roa'] = financial_enhanced['당기순이익'] / financial_enhanced['자산총계']
    financial_enhanced['roe'] = financial_enhanced['당기순이익'] / financial_enhanced['자본총계']

    # 규모 지표
    financial_enhanced['log_assets'] = np.log(financial_enhanced['자산총계'])
    financial_enhanced['log_equity'] = np.log(financial_enhanced['자본총계'])

    # 매출 기반 비율 (매출액이 있는 경우)
    financial_enhanced['has_revenue'] = financial_enhanced['매출액'].notna() & (financial_enhanced['매출액'] > 0)
    financial_enhanced['asset_turnover'] = np.where(
        financial_enhanced['has_revenue'],
        financial_enhanced['매출액'] / financial_enhanced['자산총계'],
        np.nan
    )

    # 성장률 계산 (기업별 연도별)
    growth_data = []
    for ticker in financial_enhanced['종목코드'].unique():
        company_data = financial_enhanced[financial_enhanced['종목코드'] == ticker].sort_values('year')
        if len(company_data) >= 2:
            company_data['asset_growth'] = company_data['자산총계'].pct_change()
            company_data['equity_growth'] = company_data['자본총계'].pct_change()
        growth_data.append(company_data)

    financial_final = pd.concat(growth_data, ignore_index=True)

    # 산업 규모 구분 (자산 기준 4분위)
    financial_final['size_quartile'] = pd.qcut(
        financial_final['자산총계'],
        q=4,
        labels=['Small', 'Medium_Small', 'Medium_Large', 'Large'],
        duplicates='drop'
    )

    print(f"   재무 지표 생성 완료: {financial_final.shape}")

    return financial_final

def merge_all_data(sentiment_df, financial_df, market_df):
    """전체 데이터 병합 및 MBV 계산"""

    print("\n5. 데이터 병합 및 MBV 계산")
    print("-" * 80)

    # 1단계: 재무 + 시장 데이터
    fin_market = pd.merge(
        financial_df,
        market_df,
        on=['종목코드', 'year'],
        how='inner'
    )
    print(f"   재무-시장 병합: {fin_market.shape}")

    # 2단계: 감성 데이터 추가
    final_data = pd.merge(
        fin_market,
        sentiment_df,
        on=['종목코드', 'year'],
        how='inner'
    )
    print(f"   최종 병합: {final_data.shape}")

    if len(final_data) < 100:
        print(f"경고: 표본 크기가 작습니다 (N={len(final_data)}). 최소 100개 권장.")

    # MBV 계산 및 이상치 처리
    final_data['MBV'] = final_data['market_cap_krw'] / final_data['자본총계']

    # MBV 이상치 제거 (1-99% 범위)
    mbv_q01 = final_data['MBV'].quantile(0.01)
    mbv_q99 = final_data['MBV'].quantile(0.99)
    final_data = final_data[
        (final_data['MBV'] >= mbv_q01) & (final_data['MBV'] <= mbv_q99)
    ]

    # 로그 변환 (정규성 개선)
    final_data['log_MBV'] = np.log(final_data['MBV'])

    # 추가 시장 지표
    final_data['market_liquidity'] = np.log(final_data['value_traded'] + 1)

    print(f"   MBV 계산 완료: {final_data.shape}")
    print(f"   MBV 분포: 평균={final_data['MBV'].mean():.3f}, 중위값={final_data['MBV'].median():.3f}")
    print(f"   MBV 범위: [{final_data['MBV'].min():.3f}, {final_data['MBV'].max():.3f}]")

    return final_data

def validate_data_quality(df):
    """데이터 품질 검증"""

    print("\n6. 데이터 품질 검증")
    print("-" * 80)

    # 기본 통계
    print(f"최종 분석 데이터: {df.shape}")
    print(f"기업 수: {df['종목코드'].nunique()}")
    print(f"연도 범위: {df['year'].min()}-{df['year'].max()}")
    print(f"기업당 평균 관측치: {len(df) / df['종목코드'].nunique():.1f}")

    # 핵심 변수 상관관계
    key_vars = ['sentiment_mean', 'MBV', 'log_assets', 'roa', 'debt_to_equity']
    available_vars = [var for var in key_vars if var in df.columns]

    if len(available_vars) >= 2:
        print(f"\n핵심 변수 상관관계:")
        corr_matrix = df[available_vars].corr()
        print(corr_matrix.round(4))

    # 감성-MBV 기초 관계
    if 'sentiment_mean' in df.columns:
        corr_basic, p_basic = pearsonr(df['sentiment_mean'], df['MBV'])
        print(f"\n기초 상관관계 (감성-MBV): {corr_basic:.4f} (p={p_basic:.4f})")

    return True

def run_hierarchical_regression(df):
    """위계적 회귀분석 실행 - 모델들 반환"""

    print("\n7. 위계적 회귀분석")
    print("-" * 80)

    # 변수 표준화
    scaler = StandardScaler()

    continuous_vars = ['sentiment_mean', 'log_assets', 'debt_to_equity', 'roa', 'market_liquidity']
    available_vars = [var for var in continuous_vars if var in df.columns]

    for var in available_vars:
        df[f'{var}_scaled'] = scaler.fit_transform(df[[var]])

    # 연도 더미 (중요한 연도만)
    year_counts = df['year'].value_counts()
    major_years = year_counts[year_counts >= 10].index  # 관측치 10개 이상 연도만

    for year in major_years[1:]:  # 첫 번째 연도는 기준
        df[f'year_{year}'] = (df['year'] == year).astype(int)

    print(f"분석 준비 완료: {df.shape}")
    print(f"사용 가능한 표준화 변수: {[var for var in df.columns if var.endswith('_scaled')]}")

    # 모델들 실행
    models = []
    model_names = []

    # 모델 1: 기본 모델
    try:
        model1 = smf.ols('MBV ~ sentiment_mean_scaled', data=df).fit(cov_type='HC1')
        models.append(model1)
        model_names.append('기본')
    except Exception as e:
        print(f"모델 1 실행 실패: {e}")

    # 모델 2: 기업 특성 통제
    try:
        control_vars = ['log_assets_scaled']
        if 'debt_to_equity_scaled' in df.columns:
            control_vars.append('debt_to_equity_scaled')
        if 'roa_scaled' in df.columns:
            control_vars.append('roa_scaled')

        formula2 = 'MBV ~ sentiment_mean_scaled + ' + ' + '.join(control_vars)
        model2 = smf.ols(formula2, data=df).fit(cov_type='HC1')
        models.append(model2)
        model_names.append('기업통제')
    except Exception as e:
        print(f"모델 2 실행 실패: {e}")

    # 모델 3: 시장 변수 추가
    try:
        if 'market_liquidity_scaled' in df.columns:
            formula3 = formula2 + ' + market_liquidity_scaled'
            model3 = smf.ols(formula3, data=df).fit(cov_type='HC1')
            models.append(model3)
            model_names.append('시장통제')
    except Exception as e:
        print(f"모델 3 실행 실패: {e}")

    # 모델 4: 시간 고정효과
    try:
        year_dummies = [col for col in df.columns if col.startswith('year_')]
        if year_dummies and len(models) > 0:
            # 마지막 성공한 모델의 공식에 연도 더미 추가
            last_formula = models[-1].model.formula if hasattr(models[-1].model, 'formula') else 'MBV ~ sentiment_mean_scaled + log_assets_scaled + debt_to_equity_scaled + roa_scaled + market_liquidity_scaled'
            formula4 = last_formula + ' + ' + ' + '.join(year_dummies)
            model4 = smf.ols(formula4, data=df).fit(cov_type='HC1')
            models.append(model4)
            model_names.append('시간통제')
    except Exception as e:
        print(f"모델 4 실행 실패: {e}")

    if not models:
        print("모든 모델 실행 실패")
        return None, None, None

    # 결과 비교
    print("\n모델 비교:")
    print("="*80)

    results = []
    for i, (model, name) in enumerate(zip(models, model_names)):
        sentiment_coef = model.params.get('sentiment_mean_scaled', np.nan)
        sentiment_pval = model.pvalues.get('sentiment_mean_scaled', np.nan)

        result = {
            'Model': f'{i+1}. {name}',
            'N': int(model.nobs),
            'R²': model.rsquared,
            'Adj_R²': model.rsquared_adj,
            'Sentiment_β': sentiment_coef,
            'Sentiment_p': sentiment_pval,
            'AIC': model.aic
        }
        results.append(result)

    results_df = pd.DataFrame(results)
    print(results_df.round(6))

    # 최적 모델 선택
    best_model = min(models, key=lambda x: x.aic)
    best_idx = models.index(best_model)

    print(f"\n최적 모델: {model_names[best_idx]} (AIC 기준)")
    print("="*60)
    print(best_model.summary())

    return models, model_names, best_model

def create_paper_table(models, model_names):
    """논문용 표 생성 및 저장"""

    print("\n8. 논문용 표 생성")
    print("-" * 80)

    # 변수 순서 및 이름 설정
    var_order = [
        'sentiment_mean_scaled',
        'log_assets_scaled',
        'debt_to_equity_scaled',
        'roa_scaled',
        'market_liquidity_scaled'
    ]

    # 연도 더미 변수 추가 (첫 번째 모델에서 확인)
    year_vars = []
    if models:
        for param in models[-1].params.index:  # 마지막 모델에서 연도 변수 확인
            if param.startswith('year_'):
                year_vars.append(param)
        year_vars.sort()
        var_order.extend(year_vars)

    var_rename = {
        'sentiment_mean_scaled': 'News Sentiment',
        'log_assets_scaled': 'Firm Size (log)',
        'debt_to_equity_scaled': 'Leverage',
        'roa_scaled': 'ROA',
        'market_liquidity_scaled': 'Market Liquidity'
    }

    # 연도 변수명 매핑
    for var in year_vars:
        year_num = var.replace('year_', '')
        var_rename[var] = f'Year {year_num}'

    # 모델명을 컬럼 제목으로 변환
    paper_model_names = []
    for i, name in enumerate(model_names):
        paper_model_names.append(f"({i+1})\n{name}")

    # 추가 메타정보 행
    n_models = len(models)

    add_rows = {
        "Firm Controls": ["No"] + ["Yes"] * (n_models - 1),
        "Market Controls": ["No"] * min(2, n_models) + ["Yes"] * max(0, n_models - 2),
        "Year Fixed Effects": ["No"] * min(3, n_models) + ["Yes"] * max(0, n_models - 3),
        "Standard Errors": ["Robust"] * n_models
    }

    # 실제 모델 수에 맞게 조정
    for key in add_rows:
        add_rows[key] = add_rows[key][:n_models]

    # 표 생성
    table = make_regression_table(
        results_list=models,
        model_names=paper_model_names,
        var_order=var_order,
        var_rename=var_rename,
        add_rows=add_rows,
        show_adj_r2=True
    )

    # 모든 형식으로 저장
    save_all_formats(table, "sentiment_mbv_regression")

    return table

def final_conclusion(model, df):
    """최종 결론 및 해석"""

    print("\n9. 최종 결론")
    print("="*80)

    if 'sentiment_mean_scaled' not in model.params.index:
        print("분석 실패: 감성 변수가 최종 모델에 포함되지 않음")
        return False

    coef = model.params['sentiment_mean_scaled']
    pval = model.pvalues['sentiment_mean_scaled']
    tstat = model.tvalues['sentiment_mean_scaled']
    conf_int = model.conf_int().loc['sentiment_mean_scaled']

    print(f"핵심 가설검정 결과:")
    print(f"H1: 뉴스 감성점수가 높을수록 기업가치(MBV)가 높다")
    print(f"")
    print(f"통계적 결과:")
    print(f"   - 회귀계수(β): {coef:.6f}")
    print(f"   - t-통계량: {tstat:.4f}")
    print(f"   - p-value: {pval:.6f}")
    print(f"   - 95% 신뢰구간: [{conf_int[0]:.6f}, {conf_int[1]:.6f}]")
    print(f"   - 표본 크기: {int(model.nobs)}")
    print(f"   - R-squared: {model.rsquared:.4f}")

    # 통계적 유의성 판정
    if pval < 0.01:
        significance = "1% 수준에서 통계적으로 유의"
        conclusion = "H1 강력 지지"
        success = True
    elif pval < 0.05:
        significance = "5% 수준에서 통계적으로 유의"
        conclusion = "H1 지지"
        success = True
    elif pval < 0.10:
        significance = "10% 수준에서 통계적으로 유의"
        conclusion = "H1 약한 지지"
        success = True
    else:
        significance = "통계적으로 유의하지 않음"
        conclusion = "H1 기각"
        success = False

    print(f"")
    print(f"가설검정 결론:")
    print(f"   - 통계적 유의성: {significance}")
    print(f"   - 연구 결론: {conclusion}")

    # 경제적 해석 (유의한 경우만)
    if success:
        print(f"")
        print(f"경제적 해석:")

        # 표준화 계수 해석
        sentiment_std = df['sentiment_mean'].std()
        mbv_std = df['MBV'].std()

        print(f"   - 감성점수 1 표준편차({sentiment_std:.3f}) 증가 시")
        print(f"   - MBV가 {coef:.4f} 증가 (표준화 기준)")

        # 실제 단위 해석
        actual_effect = coef * mbv_std / sentiment_std
        print(f"   - 실제 단위: 감성점수 0.1 증가 → MBV {actual_effect*0.1:.4f} 증가")

        if coef > 0:
            print(f"   - 해석: 긍정적 뉴스 감성이 기업가치를 향상시킴")
        else:
            print(f"   - 해석: 부정적 결과 (이론과 불일치)")

    print(f"")
    print(f"연구의 한계:")
    print(f"   - 표본 크기: {int(model.nobs)}개 (추가 확대 권장)")
    print(f"   - 내생성: 역인과관계 가능성")
    print(f"   - 생략변수: 관찰되지 않은 기업 특성")

    return success

def main_analysis():
    """메인 분석 실행 - 모델 결과 반환하도록 수정"""

    print("정교한 통계 분석 시작")
    print("="*100)

    try:
        # 1. 데이터 로딩 및 탐색
        news_df, financial_df, market_df = load_and_explore_data()

        # 2. 데이터 정제
        news_clean, financial_clean, market_clean = clean_and_standardize_data(
            news_df, financial_df, market_df
        )

        # 3. 변수 생성
        sentiment_features = create_sentiment_aggregates(news_clean)
        financial_features = create_financial_ratios(financial_clean)

        # 4. 데이터 병합
        final_dataset = merge_all_data(sentiment_features, financial_features, market_clean)

        # 표본 크기 확인
        if len(final_dataset) < 50:
            print(f"경고: 표본 크기가 부족합니다 (N={len(final_dataset)})")
            print("최소 50개 이상의 관측치가 권장됩니다.")
            return False, None, None, None

        # 5. 데이터 품질 검증
        validate_data_quality(final_dataset)

        # 6. 회귀분석 실행 (모델들 반환)
        models, model_names, best_model = run_hierarchical_regression(final_dataset)

        if models is None:
            print("회귀분석 실행 실패")
            return False, None, None, None

        # 7. 논문용 표 생성 및 저장
        table = create_paper_table(models, model_names)

        # 8. 최종 결론
        success = final_conclusion(best_model, final_dataset)

        print(f"\n" + "="*100)
        if success:
            print("분석 성공: 통계적으로 유의한 결과 도출")
            print("뉴스 감성이 기업가치에 미치는 영향 실증적으로 확인")
        else:
            print("분석 완료: 통계적 유의성 미달")
            print("가설 지지 증거 부족 - 추가 연구 필요")

        print("\n파일 생성 완료:")
        print("- sentiment_mbv_regression.xlsx (Excel 표)")
        print("- sentiment_mbv_regression.csv (CSV 표)")
        print("- sentiment_mbv_regression.tex (LaTeX 코드)")
        print("="*100)

        return success, models, best_model, table

    except Exception as e:
        print(f"분석 중 오류 발생: {str(e)}")
        import traceback
        traceback.print_exc()
        return False, None, None, None

# 분석 실행
if __name__ == "__main__":
    success, models, best_model, table = main_analysis()

    if success:
        print("\n연구 성과: 학술 논문 작성 및 정책 제언 가능")
        print("신뢰할 수 있는 실증 결과 확보")
        print("논문용 표가 자동으로 생성되었습니다.")
    else:
        print("\n연구 개선 방향:")
        print("1. 표본 크기 확대")
        print("2. 추가 통제변수 도입")
        print("3. 내생성 문제 해결 (도구변수 등)")
        print("4. 산업별 세분화 분析")
        print("5. 시간 지연 효과 검토")

정교한 통계 분석 시작

1. 데이터 로딩 및 기초 탐색
--------------------------------------------------------------------------------
뉴스 데이터: (12789, 18)
재무 데이터: (18701, 11)
시가총액 데이터: (21892, 9)

감성점수 분포:
평균: 0.2927, 표준편차: 0.3192
범위: [-0.771, 0.755]

2. 데이터 정제 및 표준화
--------------------------------------------------------------------------------
2-1. 뉴스 데이터 정제
   정제 후: (12745, 20)
2-2. 재무 데이터 정제
   정제 후: (16793, 12)
2-3. 시가총액 데이터 정제
   정제 후: (5355, 6)

3. 뉴스 감성 집계 (기업-연도별)
--------------------------------------------------------------------------------
   집계 결과: 1947 → 581 (뉴스 5개 이상)
   평균 뉴스 개수: 18.0

4. 재무 비율 및 지표 생성
--------------------------------------------------------------------------------
   재무 지표 생성 완료: (16793, 23)

5. 데이터 병합 및 MBV 계산
--------------------------------------------------------------------------------
   재무-시장 병합: (15046, 27)
   최종 병합: (406, 40)
   MBV 계산 완료: (396, 43)
   MBV 분포: 평균=2.215, 중위값=1.766
   MBV 범위: [0.271, 10.181]

6. 데이터 품질 검증
-------------------------------------------

In [ ]:
# 1. 데이터 로딩 및 탐색
news_df, financial_df, market_df = load_and_explore_data()

# 2. 데이터 정제
news_clean, financial_clean, market_clean = clean_and_standardize_data(
    news_df, financial_df, market_df
)

# 3. 변수 생성
sentiment_features = create_sentiment_aggregates(news_clean)
financial_features = create_financial_ratios(financial_clean)

# 4. 데이터 병합
final_dataset = merge_all_data(sentiment_features, financial_features, market_clean)
final_dataset


1. 데이터 로딩 및 기초 탐색
--------------------------------------------------------------------------------
뉴스 데이터: (12789, 18)
재무 데이터: (18701, 11)
시가총액 데이터: (21892, 9)

감성점수 분포:
평균: 0.2927, 표준편차: 0.3192
범위: [-0.771, 0.755]

2. 데이터 정제 및 표준화
--------------------------------------------------------------------------------
2-1. 뉴스 데이터 정제
   정제 후: (12745, 20)
2-2. 재무 데이터 정제
   정제 후: (16793, 12)
2-3. 시가총액 데이터 정제
   정제 후: (5355, 6)

3. 뉴스 감성 집계 (기업-연도별)
--------------------------------------------------------------------------------
   집계 결과: 1947 → 581 (뉴스 5개 이상)
   평균 뉴스 개수: 18.0

4. 재무 비율 및 지표 생성
--------------------------------------------------------------------------------
   재무 지표 생성 완료: (16793, 23)

5. 데이터 병합 및 MBV 계산
--------------------------------------------------------------------------------
   재무-시장 병합: (15046, 27)
   최종 병합: (406, 40)
   MBV 계산 완료: (396, 43)
   MBV 분포: 평균=2.215, 중위값=1.766
   MBV 범위: [0.271, 10.181]


,자산총계,부채총계,자본총계,매출액,당기순이익,bsns_year,reprt_code,report_name,report_date,기업명,...,negative_mean,negative_std,confidence_mean,confidence_std,sentiment_volatility,confidence_weighted_sentiment,news_intensity,MBV,log_MBV,market_liquidity
0,1.072148e+11,3.938808e+10,6.782669e+10,2.463637e+09,-3.684338e+09,2024,11012,반기보고서,2024-06-30,HLB파나진,...,0.396014,0.158175,0.626757,0.137438,0.316339,0.130347,0.019178,1.964762,0.675371,19.466385
1,9.154201e+10,2.873830e+10,6.280371e+10,3.407320e+09,5.712507e+08,2024,11013,1분기보고서,2024-03-31,HLB파나진,...,0.396014,0.158175,0.626757,0.137438,0.316339,0.130347,0.019178,2.121902,0.752313,19.466385
2,1.096106e+11,3.348303e+10,7.612757e+10,1.319692e+10,-2.085569e+09,2024,11011,사업보고서,2024-12-31,HLB파나진,...,0.396014,0.158175,0.626757,0.137438,0.316339,0.130347,0.019178,1.750526,0.559917,19.466385
3,1.067624e+11,3.418568e+10,7.257672e+10,3.851908e+09,-1.222960e+08,2024,11014,3분기보고서,2024-09-30,HLB파나진,...,0.396014,0.158175,0.626757,0.137438,0.316339,0.130347,0.019178,1.836172,0.607683,19.466385
4,5.648974e+11,4.037330e+11,1.611644e+11,4.584978e+11,1.730768e+10,2021,11011,사업보고서,2021-12-31,HS애드,...,0.386757,0.037370,0.613243,0.037370,0.074711,0.138891,0.019178,0.665790,-0.406781,19.133117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,5.290252e+10,8.183910e+09,4.471861e+10,1.455840e+10,-9.203877e+09,2024,11013,1분기보고서,2024-03-31,하이퍼코퍼레이션,...,0.356000,0.103032,0.645160,0.101334,0.206065,0.185802,0.082192,2.945535,1.080290,20.744028
402,5.484090e+10,9.220361e+09,4.562053e+10,1.807624e+10,1.034505e+09,2024,11012,반기보고서,2024-06-30,휴림에이텍,...,0.270500,0.091305,0.729500,0.091305,0.182609,0.334841,0.013699,0.761693,-0.272212,20.564144
403,5.383891e+10,9.107988e+09,4.473092e+10,1.782476e+10,1.315785e+09,2024,11013,1분기보고서,2024-03-31,휴림에이텍,...,0.270500,0.091305,0.729500,0.091305,0.182609,0.334841,0.013699,0.776841,-0.252519,20.564144
404,7.531147e+10,2.739607e+10,4.791540e+10,7.031696e+10,4.597622e+09,2024,11011,사업보고서,2024-12-31,휴림에이텍,...,0.270500,0.091305,0.729500,0.091305,0.182609,0.334841,0.013699,0.725212,-0.321291,20.564144


In [ ]:
final_dataset.columns

Index(['자산총계', '부채총계', '자본총계', '매출액', '당기순이익', 'bsns_year', 'reprt_code',
       'report_name', 'report_date', '기업명', '종목코드', 'year', 'debt_to_equity',
       'debt_to_assets', 'roa', 'roe', 'log_assets', 'log_equity',
       'has_revenue', 'asset_turnover', 'asset_growth', 'equity_growth',
       'size_quartile', 'market_cap_krw', 'shares_outstanding', 'volume',
       'value_traded', 'sentiment_mean', 'sentiment_std', 'news_count',
       'sentiment_median', 'positive_mean', 'positive_std', 'negative_mean',
       'negative_std', 'confidence_mean', 'confidence_std',
       'sentiment_volatility', 'confidence_weighted_sentiment',
       'news_intensity', 'MBV', 'log_MBV', 'market_liquidity'],
      dtype='object')

In [ ]:
# 변수 표준화
scaler = StandardScaler()

continuous_vars = ['sentiment_mean', 'log_assets', 'debt_to_equity', 'roa', 'market_liquidity']
available_vars = [var for var in continuous_vars if var in final_dataset.columns]

for var in available_vars:
    final_dataset[f'{var}_scaled'] = scaler.fit_transform(final_dataset[[var]])

# 연도 더미 (중요한 연도만)
year_counts = final_dataset['year'].value_counts()
major_years = year_counts[year_counts >= 10].index  # 관측치 10개 이상 연도만

for year in major_years[1:]:  # 첫 번째 연도는 기준
    final_dataset[f'year_{year}'] = (final_dataset['year'] == year).astype(int)

print(f"분석 준비 완료: {final_dataset.shape}")
print(f"사용 가능한 표준화 변수: {[var for var in final_dataset.columns if var.endswith('_scaled')]}")

분석 준비 완료: (396, 51)
사용 가능한 표준화 변수: ['sentiment_mean_scaled', 'log_assets_scaled', 'debt_to_equity_scaled', 'roa_scaled', 'market_liquidity_scaled']


#### 테이블 내보내기

In [ ]:
pd.DataFrame(final_dataset[['log_assets_scaled','sentiment_mean_scaled','debt_to_equity_scaled',
                                                'log_assets_scaled','market_liquidity_scaled','MBV','log_MBV']].corr(numeric_only=True))

,log_assets_scaled,sentiment_mean_scaled,debt_to_equity_scaled,log_assets_scaled,market_liquidity_scaled,MBV,log_MBV
log_assets_scaled,1.000000,0.025306,0.280660,1.000000,0.238695,-0.282134,-0.292856
sentiment_mean_scaled,0.025306,1.000000,-0.116460,0.025306,0.096581,-0.047332,0.000828
debt_to_equity_scaled,0.280660,-0.116460,1.000000,0.280660,-0.009666,0.178069,0.160269
log_assets_scaled,1.000000,0.025306,0.280660,1.000000,0.238695,-0.282134,-0.292856
market_liquidity_scaled,0.238695,0.096581,-0.009666,0.238695,1.000000,0.237900,0.283077
MBV,-0.282134,-0.047332,0.178069,-0.282134,0.237900,1.000000,0.908508
log_MBV,-0.292856,0.000828,0.160269,-0.292856,0.283077,0.908508,1.000000


In [ ]:
pd.DataFrame(final_dataset[['log_assets_scaled','sentiment_mean_scaled','debt_to_equity_scaled',
                                                'log_assets_scaled','market_liquidity_scaled','MBV','log_MBV']].corr(numeric_only=True)).to_csv("TABLE2_가설1_최종검정때변수_상관관계행렬.csv",encoding='utf-8-sig')